# Sprint 03 · Project 2 — Named Entity Recognition

Self-contained Kaggle notebook. Enable Internet, select GPU T4 ×2 (one GPU also works), then Save & Run All. No dataset upload is needed. Full mode is the default.

The completed original experiment trained LSTM, BiLSTM, BiLSTM + CRF and cased DistilBERT. DistilBERT was selected on validation entity F1 (93.92%) and achieved 90.18% strict entity F1 on the official test set. See README.md and the executed notebook for the recorded results.

**10 November 2026 update:** raw-text inference retains acronyms, titles and hyphenated words and handles invisible characters with original offsets. This updated notebook has static validation and local CPU regression checks; its changed source has not been retrained in Kaggle. Changing source changes experiment identity: the default run name below is new. Historical checkpoints remain evidence of the original run.

English PER, ORG, LOC and MISC only; no DATE class. The Gradio app uses an explicit Analyze action. Public hosting is a separate step.


In [ ]:
# Routine Kaggle execution needs no edits. Optional: "smoke" (32 train / 16 validation)
# or "prep" (full structural audit only). Smoke never opens/evaluates the test split.
MODE = "full"
RUN_NAME = "ner_raw_v2_" + MODE
RUNTIME_NAME = "ner_runtime_v1"

from pathlib import Path
import sys, os, json, subprocess, signal, time
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "ner_work"
BASE.mkdir(parents=True, exist_ok=True)
SOURCE = BASE / "ner_embedded_source"
ROOT = BASE / RUN_NAME
RUNTIME = BASE / RUNTIME_NAME
SOURCE.mkdir(parents=True, exist_ok=True)
assert MODE in {"full", "smoke", "prep"}
print("Remote execution directory:", BASE)
print("Mode:", MODE, "| Outputs:", ROOT / "outputs")


## Embedded implementation

The following cell writes the runtime modules and small, observed reference metadata. It contains **no corpus, vector matrix, pretrained weights, or tokenizer files**. The readable source strings can be inspected before execution. Changing code or settings invalidates completed-model reuse; use a new `RUN_NAME` for a changed experiment.


In [ ]:
EMBEDDED_FILES = {'app.py': 'import os, html\nfrom pathlib import Path\nimport gradio as gr\nfrom inference import Predictor\n\n# One source of truth for entity names and colors.\n# The colors are the only "loud" thing in the UI, so they carry the meaning.\nLABELS = {\n    \'PER\': (\'Person\', \'#6366f1\', \'#4338ca\'),\n    \'ORG\': (\'Organization\', \'#f59e0b\', \'#b45309\'),\n    \'LOC\': (\'Place\', \'#10b981\', \'#047857\'),\n    \'MISC\': (\'Other\', \'#f43f5e\', \'#be123c\'),\n}\nFALLBACK = (\'Other\', \'#64748b\', \'#334155\')\n\nEMPTY_TEXT = \'Highlighted text will appear here.\'\nHINT_EMPTY = \'Choose an example or enter some English text.\'\n\n\ndef _entity_css():\n    rules = []\n    for code, (_, color, dark_color) in LABELS.items():\n        rules.append(f\'.ent-{code}{{background:{color}26;border-bottom-color:{color};}}\')\n        rules.append(f\'.ent-{code} small{{color:{dark_color} !important;}}\')\n        rules.append(f\'.dot-{code}{{background:{color};}}\')\n    return \'\\n\'.join(rules)\n\n\nCUSTOM_CSS = f"""\nhtml {{\n    scrollbar-gutter: stable;\n}}\n.gradio-container {{\n    width: 920px !important;\n    max-width: 95vw !important;\n    margin-left: auto !important;\n    margin-right: auto !important;\n    --ink: #0f172a;\n    --on-ink: #f8fafc;\n}}\n.dark .gradio-container {{\n    --ink: #f1f5f9;\n    --on-ink: #0f172a;\n}}\n\n/* Neutral primary button: no accent color competing with the entity colors */\n.gradio-container button.primary {{\n    background: var(--ink) !important;\n    color: var(--on-ink) !important;\n    border: 1px solid var(--ink) !important;\n}}\n.gradio-container button.primary:hover {{ opacity: 0.88; }}\n\n/* Results card with clean white surface, removing muddy grey background */\n.results-card, .results-card > .styler, .results-card .styler {{\n    background: #ffffff !important;\n    border: 1px solid #e2e8f0 !important;\n    border-radius: 12px !important;\n    padding: 16px !important;\n    box-shadow: 0 1px 3px rgba(0, 0, 0, 0.04) !important;\n}}\n.results-card .status p {{\n    margin: 0;\n    color: #334155 !important;\n    font-size: 0.92rem !important;\n    font-weight: 500 !important;\n}}\n\n/* Highlighted text: clean document card */\n.ner-view {{\n    white-space: pre-wrap;\n    line-height: 2.2;\n    font-size: 1.05rem;\n    color: #0f172a;\n    padding: 12px 14px;\n    background: #f8fafc;\n    border: 1px solid #e2e8f0;\n    border-radius: 8px;\n    margin: 6px 0 12px;\n}}\n.ner-view.empty {{\n    color: #64748b;\n    border-style: dashed;\n    font-size: 0.95rem;\n}}\n.ent {{\n    padding: 2px 6px;\n    border-radius: 4px;\n    border-bottom: 2px solid transparent;\n    color: #0f172a !important;\n}}\n.ent small {{\n    margin-left: 5px;\n    font-size: 0.75em;\n    font-weight: 700;\n    letter-spacing: 0.02em;\n}}\n{_entity_css()}\n\n/* Legend */\n.legend {{ display: flex; flex-wrap: wrap; gap: 16px; margin: 0 0 10px; }}\n.legend span {{\n    display: inline-flex; align-items: center; gap: 6px;\n    font-size: 0.88rem; color: #334155; font-weight: 500;\n}}\n.legend i {{\n    width: 12px; height: 12px; border-radius: 3px;\n    display: inline-block;\n}}\n"""\n\nEMPTY_VIEW = f\'<div class="ner-view empty">{EMPTY_TEXT}</div>\'\n\n\ndef legend_html():\n    items = \'\'.join(\n        f\'<span><i class="dot-{code}"></i>{name}</span>\'\n        for code, (name, _, _) in LABELS.items()\n    )\n    return f\'<div class="legend">{items}</div>\'\n\n\ndef colored_html(text, spans):\n    out, last = [], 0\n    for s in spans:\n        name = LABELS.get(s[\'label\'], FALLBACK)[0]\n        cls = f"ent-{s[\'label\']}" if s[\'label\'] in LABELS else \'\'\n        out.append(html.escape(text[last:s[\'start\']]))\n        out.append(\n            f\'<mark class="ent {cls}" title="{name}">\'\n            + html.escape(text[s[\'start\']:s[\'end\']])\n            + f\'<small>{name}</small></mark>\'\n        )\n        last = s[\'end\']\n    out.append(html.escape(text[last:]))\n    return f\'<div class="ner-view">{"".join(out)}</div>\'\n\n\nEXAMPLES = [\n    \'Sara works at Google in Cairo.\',\n    \'Ahmed visited Dubai with Omar.\',\n    \'Apple opened an office in London.\',\n    \'An Egyptian student moved to Paris.\',\n]\n\n\ndef build_app(model_dir=None):\n    predictor = Predictor(model_dir or Path(__file__).parent / \'selected_model\')\n\n    def analyze(text):\n        if not text or not text.strip():\n            return EMPTY_VIEW, gr.update(value=[], visible=False), HINT_EMPTY\n        try:\n            result = predictor.analyze(text)\n        except ValueError as exc:\n            raise gr.Error(str(exc))\n        rows = [[x[\'text\'], LABELS.get(x[\'label\'], FALLBACK)[0]] for x in result[\'entities\']]\n        n = len(rows)\n        if n:\n            msg = f\'Found {n} \' + (\'entity.\' if n == 1 else \'entities.\')\n        else:\n            msg = \'No entities found. Try another sentence.\'\n        return colored_html(text, result[\'entities\']), gr.update(value=rows, visible=bool(n)), msg\n\n    def clear():\n        return \'\', EMPTY_VIEW, gr.update(value=[], visible=False), HINT_EMPTY\n\n    with gr.Blocks(title=\'Find entities\') as demo:\n        gr.HTML(f\'<style>{CUSTOM_CSS}</style>\')\n        gr.Markdown(\n            \'# Find entities in your text\\n\'\n            \'Paste English text to find people, organizations, places and other named entities.\'\n        )\n\n        box = gr.Textbox(\n            label=\'Your text\', lines=4, max_lines=12,\n            placeholder=\'Sara works at Google in Cairo.\',\n        )\n        for pair in (EXAMPLES[:2], EXAMPLES[2:]):\n            with gr.Row():\n                for ex in pair:\n                    gr.Button(ex, size=\'sm\', variant=\'secondary\').click(\n                        lambda ex=ex: ex, outputs=box\n                    )\n\n        with gr.Row():\n            button = gr.Button(\'Find entities\', variant=\'primary\', scale=3)\n            clear_btn = gr.Button(\'Clear\', variant=\'secondary\', scale=1)\n\n        with gr.Group(elem_classes=[\'results-card\']):\n            gr.Markdown(\'### Results\')\n            status = gr.Markdown(HINT_EMPTY, elem_classes=[\'status\'])\n            gr.HTML(legend_html())\n            view = gr.HTML(value=EMPTY_VIEW)\n            table = gr.Dataframe(\n                headers=[\'Entity\', \'Category\'], datatype=[\'str\', \'str\'],\n                interactive=False, visible=False, label=\'Detected entities\',\n            )\n\n        with gr.Accordion(\'About this demo\', open=False):\n            gr.Markdown(\n                \'This English-language demo finds **people**, **organizations**, **places** \'\n                \'and **other named entities**. Other includes nationalities, such as Egyptian.\\n\\n\'\n                \'The underlying categories are PER, ORG, LOC and MISC. The model was trained on \'\n                \'English news articles and can make mistakes. Dates are not a separate supported category.\\n\\n\'\n                \'You can enter up to **20,000 characters**. Your original spacing and punctuation \'\n                \'are preserved. Text is processed when you select Find entities.\'\n            )\n\n        button.click(analyze, inputs=box, outputs=[view, table, status],\n                     concurrency_limit=1, concurrency_id=\'inference\')\n        clear_btn.click(clear, outputs=[box, view, table, status])\n    return demo.queue(max_size=8, default_concurrency_limit=1)\n\n\nif __name__ == \'__main__\':\n    build_app().launch(server_name=\'127.0.0.1\', server_port=int(os.environ.get(\'PORT\', \'7860\')))\n', 'bootstrap.py': '"""Install first, validate in a fresh process. Never restart/kill the notebook kernel."""\nimport sys,os,subprocess,json,venv,sysconfig,importlib.util,importlib.metadata as im\nfrom pathlib import Path\nos.environ[\'MPLBACKEND\']=\'Agg\'\n\ndef main(env_path,source):\n    env_path=Path(env_path).resolve();source=Path(source).resolve()\n    try:\n        import torch\n    except ImportError as exc:\n        raise RuntimeError(\'The host needs working Kaggle CUDA PyTorch. Select a Kaggle GPU accelerator, then Run All. No torch downgrade was attempted.\') from exc\n    from packaging.version import Version\n    if Version(torch.__version__.split(\'+\')[0])<Version(\'2.4.0\'):\n        raise RuntimeError(f\'Host torch {torch.__version__} is too old; use a current Kaggle image. Host torch was not changed.\')\n    host={\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'python\':sys.version,\'host_executable\':sys.executable}\n    marker=env_path/\'host_torch.json\'\n    if marker.exists() and json.loads(marker.read_text())!=host:\n        raise RuntimeError(\'Isolated runtime belongs to a different host Python/PyTorch. Use a new RUNTIME_NAME in the notebook settings.\')\n    # Avoid ensurepip: host pip seeds an isolated pip-less environment.\n    # Also repairs a partially created runtime left by a failed ensurepip attempt.\n    if not (env_path/\'bin/python\').exists():\n        venv.EnvBuilder(with_pip=False,system_site_packages=False).create(env_path)\n    py=env_path/\'bin/python\'\n    import pip\n    if Version(pip.__version__) < Version(\'22.3\'):\n        raise RuntimeError(\'Host pip >=22.3 required for --python targeting. Use a current Kaggle image; host packages were not changed.\')\n    print(\'Seeding isolated pip without ensurepip; host packages are preserved.\',flush=True)\n    subprocess.run([sys.executable,\'-m\',\'pip\',\'--python\',str(py),\'install\',\n                    \'--disable-pip-version-check\',\'pip==25.3\',\'setuptools==80.9.0\',\'wheel==0.45.1\',\'wrapt==1.17.3\'],check=True)\n    subprocess.run([str(py),\'-m\',\'pip\',\'--version\'],check=True)\n    site=Path(subprocess.check_output([str(py),\'-c\',\'import sysconfig;print(sysconfig.get_paths()["purelib"])\'],text=True).strip())\n    # Expose only host PyTorch/CUDA components, not preloaded vision/audio integrations.\n    for name in [\'torch\',\'torchgen\',\'functorch\',\'triton\',\'nvidia\',\'cuda\']:\n        spec=importlib.util.find_spec(name)\n        if spec is None:continue\n        paths=list(spec.submodule_search_locations or [])\n        if paths:\n            original=Path(paths[0]);target=site/name\n            if not target.exists():target.symlink_to(original,target_is_directory=True)\n    selected=[]\n    for dist in im.distributions():\n        name=dist.metadata[\'Name\'].lower().replace(\'_\',\'-\')\n        if name in {\'torch\',\'triton\'} or name.startswith((\'nvidia-\',\'cuda-\',\'pytorch-triton\')):\n            origin=Path(dist._path);target=site/origin.name\n            if not target.exists():target.symlink_to(origin,target_is_directory=True)\n            selected.append(name)\n    print(\'Installing isolated text runtime; preserving host CUDA PyTorch\',host,flush=True)\n    subprocess.run([str(py),\'-m\',\'pip\',\'install\',\'--disable-pip-version-check\',\'--quiet\',\'-r\',str(source/\'requirements.txt\')],check=True)\n    env=os.environ.copy();env.update({\'OMP_NUM_THREADS\':\'2\',\'TOKENIZERS_PARALLELISM\':\'false\'})\n    code="""import torch,transformers,accelerate,seqeval,pyarrow,gradio,json\nfrom transformers import Trainer,AutoTokenizer,AutoModelForTokenClassification,TrainingArguments\nprint(json.dumps({\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'transformers\':transformers.__version__,\'accelerate\':accelerate.__version__,\'gradio\':gradio.__version__,\'cuda_devices\':torch.cuda.device_count()}))"""\n    result=subprocess.check_output([str(py),\'-c\',code],env=env,text=True)\n    print(result,flush=True);got=json.loads(result.strip().splitlines()[-1]);assert got[\'torch\']==host[\'torch\'] and got[\'cuda\']==host[\'cuda\']\n    subprocess.run([str(py),\'-m\',\'pip\',\'check\'],check=True)\n    marker.write_text(json.dumps(host,indent=2));print(\'Runtime ready. No kernel restart required.\',flush=True)\nif __name__==\'__main__\':main(sys.argv[1],sys.argv[2])\n', 'common.py': '"""Shared strict scoring and deterministic text policies. No prediction BIO repair."""\nimport json, hashlib, re, os, unicodedata\nfrom pathlib import Path\nfrom collections import Counter\nTYPES=(\'PER\',\'ORG\',\'LOC\',\'MISC\')\ndef read_json(p):\n    return json.loads(Path(p).read_text(encoding="utf-8"))\ndef write_json(p,obj):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_suffix(p.suffix+\'.tmp\'); tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False),encoding="utf-8"); tmp.replace(p)\ndef digest(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(\',\',\':\'),ensure_ascii=False).encode()).hexdigest()\ndef file_hash(p,method=\'sha256\'):\n    h=hashlib.new(method)\n    with open(p,\'rb\') as f:\n        while b:=f.read(1024*1024): h.update(b)\n    return h.hexdigest()\ndef normalize_iob(tags):\n    """IOB1 -> IOB2: orphan/type-changing I-X becomes B-X; retain every valid B-X."""\n    result=[]; conversions=[]; previous=\'O\'\n    for i,t in enumerate(tags):\n        if t!=\'O\' and not re.fullmatch(r\'[BI]-(PER|ORG|LOC|MISC)\',t): raise ValueError(f\'Malformed tag {t}\')\n        new=t\n        if t.startswith(\'I-\') and (previous==\'O\' or previous[2:]!=t[2:]):\n            new=\'B-\'+t[2:]; conversions.append({\'word\':i,\'from\':t,\'to\':new})\n        result.append(new); previous=new\n    return result,conversions\n\ndef strict_spans(tags):\n    """Match seqeval strict IOB2: orphan I is ignored; never synthesize B predictions."""\n    out=[]; start=None; typ=None\n    for i,t in enumerate(list(tags)+[\'O\']):\n        if start is not None and t!=\'I-\'+typ:\n            out.append((typ,start,i)); start=None;typ=None\n        if t.startswith(\'B-\'): typ=t[2:];start=i\n    return out\n\ndef metrics(gold,pred):\n    from seqeval.metrics import precision_score,recall_score,f1_score,classification_report,accuracy_score\n    from seqeval.scheme import IOB2\n    assert len(gold)==len(pred) and all(len(a)==len(b) for a,b in zip(gold,pred))\n    kw={\'mode\':\'strict\',\'scheme\':IOB2,\'zero_division\':0}\n    report=classification_report(gold,pred,output_dict=True,**kw)\n    per={t:report.get(t,{\'precision\':0.,\'recall\':0.,\'f1-score\':0.,\'support\':0}) for t in TYPES}\n    per={t:{k:int(v) if k==\'support\' else float(v) for k,v in row.items()} for t,row in per.items()}\n    # Independent exact-span cross-check, including illegal I transitions.\n    gs={(i,*s) for i,row in enumerate(gold) for s in strict_spans(row)}\n    ps={(i,*s) for i,row in enumerate(pred) for s in strict_spans(row)}\n    exact=2*len(gs&ps)/(len(gs)+len(ps)) if gs or ps else 0.\n    f=float(f1_score(gold,pred,**kw)); assert abs(exact-f)<1e-10,(exact,f)\n    return {\'precision\':float(precision_score(gold,pred,**kw)),\'recall\':float(recall_score(gold,pred,**kw)),\n            \'f1\':f,\'macro_f1\':sum(per[t][\'f1-score\'] for t in TYPES)/4,\n            \'token_accuracy\':float(accuracy_score(gold,pred)), \'per_type\':per,\n            \'sentences\':len(gold),\'tokens\':sum(map(len,gold))}\n\ndef invalid_bio(tags):\n    prev=\'O\';n=0\n    for t in tags:\n        n+=int(t.startswith(\'I-\') and (prev==\'O\' or prev[2:]!=t[2:]));prev=t\n    return n\n\n# Raw-text policy only; benchmark token lists are never retokenized.\nTOKEN_PATTERN=r"(?:[^\\W\\d_]\\.){2,}|(?i:Mr|Mrs|Ms|Dr|Prof|Sr|Jr|St|vs|etc)\\.|\\w+(?:[\'\\u2019\\-\\u2010\\u2011]\\w+)*|[^\\w\\s]"\nAPP_TOKENIZER_POLICY=\'raw-v2: acronyms/titles and hyphenated words retained; format/control characters omitted from inference with original offsets preserved; standalone .!? and newlines split sentences\'\n\ndef text_sentences(text):\n    # Map the inference view back to the untouched input, including internal gaps.\n    positions=[i for i,c in enumerate(text) if unicodedata.category(c) not in (\'Cf\',\'Cc\') or c in \'\\t\\n\\r\']\n    view=\'\'.join(text[i] for i in positions)\n    result=[];current=[]\n    matches=list(re.finditer(TOKEN_PATTERN,view,re.UNICODE))\n    for i,m in enumerate(matches):\n        current.append((m.group(),positions[m.start()],positions[m.end()-1]+1))\n        nxt=matches[i+1].start() if i+1<len(matches) else len(view)\n        gap=view[m.end():nxt]\n        if \'\\n\' in gap or (m.group() in \'.!?\' and (gap or i+1==len(matches))):\n            result.append(current);current=[]\n    if current:result.append(current)\n    return result\n\ndef highlight_spans(text,words,tags):\n    assert len(words)==len(tags)\n    spans=[]\n    for typ,a,b in strict_spans(tags):\n        start,end=words[a][1],words[b-1][2]\n        spans.append({\'start\':start,\'end\':end,\'label\':typ,\'text\':text[start:end]})\n    return spans\n\ndef word_chunks(tokens,tokenizer,max_length):\n    """Greedy, label-independent whole-word chunks. No dropped words or subword targets.\n    Each word must fit; BERT\'s single-word [UNK] behavior is allowed and audited.\n    Chunking never consults gold labels. Reconstruction precedes strict scoring.\n    """\n    lengths=[len(tokenizer(t,add_special_tokens=False)[\'input_ids\']) for t in tokens]\n    assert all(n>0 for n in lengths),\'Tokenizer produced zero subwords\'\n    budget=max_length-tokenizer.num_special_tokens_to_add(pair=False)\n    out=[];start=0;used=0\n    for i,n in enumerate(lengths):\n        if n>budget:raise ValueError(f\'One word exceeds transformer capacity at {i}; no truncation permitted\')\n        if used+n>budget:out.append((start,i));start=i;used=0\n        used+=n\n    if tokens:out.append((start,len(tokens)))\n    assert sum(b-a for a,b in out)==len(tokens)\n    return out,lengths\n\ndef aligned_features(records,tokenizer,labels,max_length):\n    label2id={x:i for i,x in enumerate(labels)};features=[];mapping=[]\n    for sid,row in enumerate(records):\n        chunks,_=word_chunks(row[\'tokens\'],tokenizer,max_length)\n        for a,b in chunks:\n            enc=tokenizer(row[\'tokens\'][a:b],is_split_into_words=True,add_special_tokens=True,truncation=False)\n            assert len(enc[\'input_ids\'])<=max_length\n            ids=enc.word_ids(); prev=None; targets=[];first=[]\n            for pos,w in enumerate(ids):\n                if w is not None and w!=prev:\n                    targets.append(label2id[row[\'tags\'][a+w]]);first.append((pos,a+w))\n                else:targets.append(-100)\n                prev=w\n            assert [w for p,w in first]==list(range(a,b))\n            feature={k:v for k,v in enc.items() if k in (\'input_ids\',\'attention_mask\')};feature[\'labels\']=targets\n            features.append(feature);mapping.append({\'sentence\':sid,\'a\':a,\'b\':b,\'first\':first})\n    return features,mapping\n\ndef reconstruct(chunk_predictions,mapping,records,labels):\n    result=[[None]*len(r[\'tokens\']) for r in records]\n    assert len(chunk_predictions)==len(mapping)\n    for ids,mp in zip(chunk_predictions,mapping):\n        for pos,w in mp[\'first\']:\n            assert result[mp[\'sentence\']][w] is None\n            result[mp[\'sentence\']][w]=labels[int(ids[pos])]\n    assert all(all(t is not None for t in row) for row in result)\n    return result\n', 'finalize.py': '"""Freeze validation selection, then (full mode only) test all four, export and package."""\nimport os,sys,time,shutil,zipfile,subprocess,platform\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom common import *\nfrom scheduler import MODELS,completed\nfrom worker import recurrent_predictions,transformer_predict,device_setup\nfrom models import load_recurrent\n\ndef diagnostics(records,preds):\n    counts=Counter();examples=[]\n    for r,p in zip(records,preds):\n        gold=set(strict_spans(r[\'tags\']));guess=set(strict_spans(p));counts[\'gold\']+=len(gold);counts[\'predicted\']+=len(guess);counts[\'exact\']+=len(gold&guess)\n        counts[\'invalid_bio_transitions\']+=invalid_bio(p)\n        for typ,a,b in gold-guess:\n            overlaps=[s for s in guess if max(a,s[1])<min(b,s[2])]\n            if any((a,b)==(x,y) for t,x,y in overlaps):category=\'type_confusion\'\n            elif any(typ==t for t,x,y in overlaps):category=\'boundary_error\'\n            else:category=\'missed_entity\'\n            counts[category]+=1\n        counts[\'spurious_entities\']+=sum(not any(max(a,x)<min(b,y) for t,x,y in gold) for typ,a,b in guess-gold)\n        counts[\'multiword_gold\']+=sum(b-a>1 for t,a,b in gold)\n        counts[\'multiword_exact\']+=sum(b-a>1 for t,a,b in gold&guess)\n        if gold!=guess and len(examples)<60:\n            examples.append({\'id\':r[\'id\'],\'tokens\':r[\'tokens\'],\'gold\':r[\'tags\'],\'prediction\':p,\'gold_spans\':sorted(gold),\'predicted_spans\':sorted(guess)})\n    return {\'counts\':dict(counts),\'definitions\':\'Strict IOB2 spans without label repair. Error categories count gold errors; spurious means no gold overlap. Categories are diagnostic, not replacements for exact F1.\',\'examples\':examples}\n\ndef crf_pairs(records,bilstm,crf):\n    counts=Counter();corrections=[];regressions=[];boundary_corrections=[];boundary_regressions=[]\n    for r,a,b in zip(records,bilstm,crf):\n        g=set(strict_spans(r[\'tags\']));aa=set(strict_spans(a));bb=set(strict_spans(b))\n        fix=(g&bb)-aa;reg=(g&aa)-bb\n        counts[\'exact_entity_corrections\']+=len(fix);counts[\'exact_entity_regressions\']+=len(reg)\n        counts[\'bilstm_invalid_bio\']+=invalid_bio(a);counts[\'crf_invalid_bio\']+=invalid_bio(b)\n        counts[\'bilstm_spurious\']+=len(aa-g);counts[\'crf_spurious\']+=len(bb-g)\n        gb={(x,y) for t,x,y in g};ab={(x,y) for t,x,y in aa};cb={(x,y) for t,x,y in bb}\n        counts[\'boundary_only_corrections\']+=len((gb&cb)-ab);counts[\'boundary_only_regressions\']+=len((gb&ab)-cb)\n        example={\'id\':r[\'id\'],\'tokens\':r[\'tokens\'],\'gold\':r[\'tags\'],\'bilstm\':a,\'bilstm_crf\':b,\'corrected\':sorted(fix),\'regressed\':sorted(reg)}\n        if fix and len(corrections)<30:corrections.append(example)\n        if reg and len(regressions)<30:regressions.append(example)\n    return {\'counts\':dict(counts),\'correction_examples\':corrections,\'regression_examples\':regressions,\n            \'interpretation\':\'Paired exact-span changes and type-agnostic boundary changes; unconstrained CRF is not guaranteed to win. Identical strict scoring with no BIO repair.\'}\n\ndef plot_results(root,results):\n    import os;os.environ[\'MPLBACKEND\']=\'Agg\'\n    import matplotlib;matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,axes=plt.subplots(1,3,figsize=(15,4))\n    for name,r in results.items():\n        h=r[\'history\'];v=[x for x in h if \'val_f1\' in x];tr=[x for x in h if \'train_loss\' in x];vl=[x for x in h if \'val_loss\' in x]\n        axes[0].plot([x[\'epoch\'] for x in v],[x[\'val_f1\'] for x in v],marker=\'.\',label=name)\n        if tr:axes[1].plot([x[\'epoch\'] for x in tr],[x[\'train_loss\'] for x in tr],label=name+\' train\')\n        if vl:axes[1].plot([x[\'epoch\'] for x in vl],[x[\'val_loss\'] for x in vl],linestyle=\'--\',label=name+\' val\')\n    names=list(results);axes[2].bar(names,[results[n][\'validation\'][\'f1\'] for n in names]);axes[2].tick_params(axis=\'x\',rotation=30)\n    axes[0].set_title(\'Validation strict entity micro F1\');axes[0].legend(fontsize=8)\n    axes[1].set_title(\'Loss (CRF sequence vs token CE differ)\');axes[1].legend(fontsize=6);axes[2].set_title(\'Best validation checkpoints\')\n    fig.tight_layout();fig.savefig(root/\'reports/learning_curves.png\',dpi=130);plt.close(fig)\n\ndef export_model(root,winner,result):\n    target=root/\'app\';target.mkdir(exist_ok=True)\n    source=Path(__file__).parent\n    for name in [\'common.py\',\'models.py\',\'inference.py\',\'app.py\']:shutil.copy2(source/name,target/name)\n    modeldir=target/\'selected_model\'\n    if modeldir.exists():shutil.rmtree(modeldir)\n    modeldir.mkdir();shutil.copy2(root/\'prepared/vocab.json\',modeldir/\'vocab.json\')\n    if winner==\'distilbert\':shutil.copytree(root/\'models\'/winner/\'best\',modeldir/\'model\')\n    else:shutil.copy2(root/\'models\'/winner/\'best.pt\',modeldir/\'best.pt\')\n    meta={\'model\':winner,\'labels\':read_json(root/\'prepared/vocab.json\')[\'labels\'],\'length_policy\':read_json(root/\'prepared/length_policy.json\'),\n          \'selection\':read_json(root/\'reports/selection_frozen.json\'),\'experiment_id\':result[\'experiment_id\'],\n          \'preprocessing\':\'Preserve casing/punctuation/word order; train-only vocab; lowercase only external GloVe lookup. No POS/chunk features.\',\n          \'span_policy\':\'Strict IOB2; invalid I predictions ignored rather than repaired. Half-open original character offsets.\',\n          \'app_tokenizer\':APP_TOKENIZER_POLICY,\'ui_revision\':\'polished-v2\',\n          \'benchmark_tokenization_note\':\'Raw-text regex is deterministic but differs from human CoNLL tokenization; exported token API preserves benchmark tokens exactly.\',\n          \'embedding_provenance\':read_json(source/\'source_lock.json\')[\'embedding_metadata\'],\n          \'training_mode\':result[\'mode\'],\'full_training_results\':result[\'mode\']==\'full\'}\n    write_json(modeldir/\'export.json\',meta)\n    req=source.joinpath(\'requirements.txt\').read_text()\n    target.joinpath(\'requirements.txt\').write_text(\'torch>=2.4,<3\\n\'+req)\n    target.joinpath(\'README.md\').write_text(\'# English NER Gradio app\\n\\n\'+(\'FUNCTIONAL SMOKE MODEL ONLY. Not a final trained deployment.\\n\\n\' if result[\'mode\']==\'smoke\' else \'\')+\n        \'Create a Python 3.11 or 3.12 environment, install `pip install -r requirements.txt`, then run `python app.py`. CPU works. GPU hosting is optional. No training corpus or GloVe resource is downloaded. Model/tokenizer reads are local-only.\\n\\n\'\n        \'English PER, ORG, LOC and MISC. Exact source text, end-exclusive offsets, blank/no-entity and length-limit handling. The raw-v2 tokenizer retains acronyms, common titles, apostrophes and internal hyphens. Invisible format/control characters are omitted only from inference; display text and source offsets are preserved. Tokenization remains heuristic. Publication is a separate action.\\n\')\n    return target\n\ndef archive_tree(path,destination):\n    with zipfile.ZipFile(destination,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:\n        for p in sorted(Path(path).rglob(\'*\')):\n            if p.is_file() and \'__pycache__\' not in p.parts:z.write(p,str(p.relative_to(path)))\n    with zipfile.ZipFile(destination) as z:assert z.testzip() is None\n\ndef main(root,mode=\'full\'):\n    root=Path(root);device=device_setup();reports=root/\'reports\';reports.mkdir(exist_ok=True)\n    exp=read_json(root/\'experiment.json\');identity=exp[\'id\']\n    for name in MODELS:assert completed(root,name,identity)\n    results={n:read_json(root/\'models\'/n/\'result.json\') for n in MODELS}\n    # Selection and tie rule are independent of all test predictions.\n    best=max(r[\'validation\'][\'f1\'] for r in results.values())\n    eligible=[n for n in MODELS if best-results[n][\'validation\'][\'f1\']<=.001+1e-12]\n    winner=min(eligible,key=lambda n:(results[n][\'trainable_parameters\'],results[n][\'validation_seconds\'],n))\n    frozen={\'winner\':winner,\'best_validation_f1\':best,\'eligible_within_0.001\':eligible,\n        \'rule\':\'Strict validation micro F1; within absolute 0.001 prefer fewer trainable parameters, then measured validation seconds, then stable model name\',\n        \'experiment_id\':identity,\'mode\':mode,\'checkpoint_manifests\':{n:file_hash(root/\'models\'/n/\'done.json\') for n in MODELS}}\n    freeze_path=reports/\'selection_frozen.json\'\n    if freeze_path.exists():assert read_json(freeze_path)==frozen,\'Frozen selection changed; stop test evaluation\'\n    else:write_json(freeze_path,frozen)\n    vocab=read_json(root/\'prepared/vocab.json\');val=read_json(root/\'prepared/validation.json\')\n    if mode==\'smoke\':val=val[:16]\n    vp={n:read_json(root/\'models\'/n/\'validation_predictions.json\') for n in MODELS}\n    for n in MODELS:\n        assert metrics([r[\'tags\'] for r in val],vp[n])==results[n][\'validation\']\n        write_json(reports/(n+\'_validation_diagnostics.json\'),diagnostics(val,vp[n]))\n    write_json(reports/\'crf_validation_comparison.json\',crf_pairs(val,vp[\'bilstm\'],vp[\'bilstm_crf\']))\n    test_results={};test_preds={}\n    if mode==\'full\':\n        # FIRST MODEL ACCESS TO TEST: only after the frozen selection file is on disk.\n        test=read_json(root/\'prepared/test.json\')\n        for name in MODELS:\n            if name==\'distilbert\':\n                from transformers import AutoTokenizer,AutoModelForTokenClassification\n                path=root/\'models\'/name/\'best\';tok=AutoTokenizer.from_pretrained(path,local_files_only=True)\n                model=AutoModelForTokenClassification.from_pretrained(path,local_files_only=True).to(device).eval()\n                pred,seconds=transformer_predict(model,tok,test,vocab[\'labels\'],read_json(root/\'prepared/length_policy.json\')[\'max_subwords\'],device)\n            else:\n                model=load_recurrent(root/\'models\'/name/\'best.pt\',device);pred,seconds,_=recurrent_predictions(model,test,vocab,device)\n            test_results[name]={**metrics([r[\'tags\'] for r in test],pred),\'inference_seconds\':seconds};test_preds[name]=pred\n            write_json(reports/(name+\'_test_predictions.json\'),[{\'id\':r[\'id\'],\'tags\':p} for r,p in zip(test,pred)])\n            write_json(reports/(name+\'_test_diagnostics.json\'),diagnostics(test,pred));del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n        write_json(reports/\'crf_test_comparison.json\',crf_pairs(test,test_preds[\'bilstm\'],test_preds[\'bilstm_crf\']))\n    comparison={\'mode\':mode,\'winner\':winner,\'validation\':{n:r[\'validation\'] for n,r in results.items()},\'test\':test_results,\n                \'experiments\':results,\'selection_used_test\':False,\'prediction_repair\':\'None; strict IOB2 for every architecture\'}\n    write_json(reports/\'comparison.json\',comparison);plot_results(root,results)\n    app=export_model(root,winner,results[winner])\n    sys.path.insert(0,str(app))\n    from inference import Predictor\n    predictor=Predictor(app/\'selected_model\');export_val=val[:2]\n    exact=predictor.predict_tokens([r[\'tokens\'] for r in export_val]);assert exact==vp[winner][:2]\n    assert predictor.analyze(\'  \')[\'entities\']==[]\n    for raw in [\'Alice\\u200bSmith works at Microsoft.\',\'👩\\u200d💻 Alice lives in London.\',\'Micro\\u00adsoft is in London.\',\'\\ufeffAlice lives in London.\']:\n        checked=predictor.analyze(raw)\n        assert checked[\'text\']==raw\n        assert all(raw[x[\'start\']:x[\'end\']]==x[\'text\'] for x in checked[\'entities\'])\n\n    analyzed=predictor.analyze(\'Alice works at Microsoft in London.\\nAlice visited New York.\')\n    assert all(analyzed[\'text\'][s[\'start\']:s[\'end\']]==s[\'text\'] for s in analyzed[\'entities\'])\n    no_entities=predictor.analyze(\'12345 + 67890 = 80235.\');assert isinstance(no_entities[\'entities\'],list)\n    long=\'Alice in London. \'*300;long_result=predictor.analyze(long)\n    try:predictor.analyze(\'a\'*20001);raise AssertionError(\'Long input must be rejected explicitly\')\n    except ValueError:pass\n    # Exercise actual Gradio component construction with the clean export.\n    from app import build_app\n    demo=build_app(app/\'selected_model\');assert demo is not None\n    verification={\'clean_cpu_export_reload_matches_validation\':True,\'blank\':True,\'no_entity_response\':True,\n        \'long_input_no_truncation\':True,\'over_limit_rejected\':True,\'original_offsets_valid\':True,\'gradio_components_constructed\':True,\n        \'mode\':mode,\'test_evaluated\':mode==\'full\',\'actual_cuda\':device.type==\'cuda\',\'training_full\':mode==\'full\',\n        \'simulated_scheduler_checks\':\'See scheduler_verification.json; simulation is not native two-GPU proof\'}\n    write_json(reports/\'export_verification.json\',verification)\n    env={\'python\':sys.version,\'platform\':platform.platform(),\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\n         \'pip_freeze\':subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True).splitlines()}\n    write_json(reports/\'environment.json\',env)\n    report_text=\'# NER experiment report\\n\\n\'+(\'SMOKE ONLY: 32 train / 16 validation sentences; no held-out test inference and no quality claims.\\n\\n\' if mode==\'smoke\' else \'Four best validation checkpoints evaluated on the official test split after freezing the winner.\\n\\n\')\n    report_text+=\'Selected model: **\'+winner+\'** using the predeclared validation rule.\\n\\n| Model | Validation F1 | Test F1 | Parameters | Train seconds | Peak CUDA allocated MB |\\n|---|---:|---:|---:|---:|---:|\\n\'\n    for n,r in results.items():report_text+=f"| {n} | {r[\'validation\'][\'f1\']:.6f} | {test_results[n][\'f1\'] if n in test_results else \'not evaluated\'} | {r[\'parameters\']} | {r[\'train_seconds\']:.1f} | {(r[\'cuda_max_allocated_bytes\'] or 0)/1e6:.1f} |\\n"\n    report_text+=\'\\nSee comparison.json for per-type precision, recall, F1, support, timing, checkpoint sizes, gradients and learning history; CRF comparison files contain both corrections and regressions. Concurrent timing is not an isolated speed benchmark. Losses differ between CRF and cross-entropy and should not be directly ranked.\\n\\n\'\n    report_text+=\'No optimizer-state resume is implemented. Completed compatible models are reusable after manifest verification. Interrupted models restart. Smoke and full have separate experiment identities.\\n\'\n    reports.joinpath(\'REPORT.md\').write_text(report_text)\n    outputs=root/\'outputs\';outputs.mkdir(exist_ok=True)\n    # Stage complete experiment evidence without full training inputs or external assets.\n    evidence=root/\'experiment_export\'\n    if evidence.exists():shutil.rmtree(evidence)\n    evidence.mkdir();shutil.copytree(root/\'models\',evidence/\'models\');shutil.copytree(reports,evidence/\'reports\')\n    src=evidence/\'source\';src.mkdir()\n    for p in Path(__file__).parent.iterdir():\n        if p.is_file():shutil.copy2(p,src/p.name)\n    for name in [\'vocab.json\',\'length_policy.json\',\'identity.json\']:shutil.copy2(root/\'prepared\'/name,evidence/name)\n    for name in [\'experiment.json\',\'schedule_events.json\',\'reuse_verification.json\']:shutil.copy2(root/name,evidence/name)\n    if (root/\'logs\').exists():shutil.copytree(root/\'logs\',evidence/\'worker_logs\')\n    prefix=\'smoke_\' if mode==\'smoke\' else \'\'\n    archive_tree(reports,outputs/(prefix+\'ner_reports.zip\'));archive_tree(evidence,outputs/(prefix+\'ner_all_checkpoints.zip\'));archive_tree(app,outputs/(prefix+\'ner_gradio_app.zip\'))\n    # Deployment must contain neither full GloVe nor corpus nor prepared embeddings.\n    with zipfile.ZipFile(outputs/(prefix+\'ner_gradio_app.zip\')) as z:\n        assert not any(n.endswith((\'.parquet\',\'.npy\')) or \'glove-wiki\' in n for n in z.namelist())\n    with zipfile.ZipFile(outputs/(prefix+\'ner_all_checkpoints.zip\')) as z:\n        names=z.namelist()\n        for n in MODELS:assert any(x.startswith(\'models/\'+n+\'/\') and (x.endswith(\'best.pt\') or x.endswith(\'model.safetensors\')) for x in names)\n    write_json(outputs/\'archive_manifest.json\',{p.name:{\'bytes\':p.stat().st_size,\'sha256\':file_hash(p)} for p in outputs.glob(\'*.zip\')})\n    print(\'FINISHED\',json.dumps({\'mode\':mode,\'winner\':winner,\'outputs\':str(outputs),\'test_evaluated\':mode==\'full\'}),flush=True)\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--mode\',default=\'full\');a=p.parse_args();main(a.root,a.mode)\n', 'inference.py': '"""Clean CPU inference; only exported assets, no dataset or embedding downloads."""\nimport os\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\',\'false\')\nfrom pathlib import Path\nimport torch\nfrom common import read_json,text_sentences,highlight_spans,word_chunks,strict_spans\nfrom models import load_recurrent,collate_recurrent\n\nclass Predictor:\n    def __init__(self,path,device=\'cpu\'):\n        self.path=Path(path);self.meta=read_json(self.path/\'export.json\');self.vocab=read_json(self.path/\'vocab.json\')\n        self.length=self.meta[\'length_policy\'];self.kind=self.meta[\'model\'];self.device=torch.device(device)\n        torch.set_num_threads(2)\n        if self.kind==\'distilbert\':\n            from transformers import AutoModelForTokenClassification,AutoTokenizer\n            self.tokenizer=AutoTokenizer.from_pretrained(self.path/\'model\',use_fast=True,local_files_only=True)\n            self.model=AutoModelForTokenClassification.from_pretrained(self.path/\'model\',local_files_only=True).to(self.device).eval()\n        else:self.model=load_recurrent(self.path/\'best.pt\',self.device)\n    @torch.inference_mode()\n    def predict_tokens(self,sentences):\n        out=[]\n        for words in sentences:\n            if not words:out.append([]);continue\n            if self.kind==\'distilbert\':\n                chunks,_=word_chunks(words,self.tokenizer,self.length[\'max_subwords\']);tags=[]\n                for a,b in chunks:\n                    enc=self.tokenizer(words[a:b],is_split_into_words=True,return_tensors=\'pt\',truncation=False)\n                    ids=enc.word_ids();values=self.model(**{k:v.to(self.device) for k,v in enc.items() if k in (\'input_ids\',\'attention_mask\')}).logits.argmax(-1)[0].tolist()\n                    previous=None\n                    for i,w in enumerate(ids):\n                        if w is not None and w!=previous:tags.append(self.vocab[\'labels\'][values[i]])\n                        previous=w\n            else:\n                tags=[];cap=self.length[\'app_recurrent_chunk_words\']\n                for a in range(0,len(words),cap):\n                    row={\'tokens\':words[a:a+cap]};batch=collate_recurrent([row],self.vocab)\n                    batch={k:v.to(self.device) for k,v in batch.items()};e=self.model(batch[\'words\'],batch[\'chars\'],batch[\'mask\'])\n                    tags.extend(self.vocab[\'labels\'][i] for i in self.model.decode(e,batch[\'mask\'])[0])\n            assert len(tags)==len(words);out.append(tags)\n        return out\n    def analyze(self,text):\n        if not isinstance(text,str):raise ValueError(\'Please enter English text.\')\n        if not text.strip():return {\'text\':text,\'entities\':[],\'status\':\'Enter English text to analyze.\'}\n        if len(text)>self.length[\'app_max_characters\']:\n            raise ValueError(f"Input has {len(text):,} characters; maximum {self.length[\'app_max_characters\']:,}. Divide it into smaller passages; nothing was truncated.")\n        sentences=text_sentences(text)\n        if self.kind==\'distilbert\':\n            # Some standalone Unicode marks normalize to zero BERT subwords.\n            # Omit these only in raw-text inference, retaining original display offsets.\n            sentences=[[w for w in s if self.tokenizer(w[0],add_special_tokens=False)[\'input_ids\']] for s in sentences]\n            sentences=[s for s in sentences if s]\n        preds=self.predict_tokens([[x[0] for x in s] for s in sentences])\n        spans=[]\n        for words,tags in zip(sentences,preds):spans.extend(highlight_spans(text,words,tags))\n        for s in spans:assert text[s[\'start\']:s[\'end\']]==s[\'text\']\n        return {\'text\':text,\'entities\':spans,\'status\':f\'{len(spans)} entities found.\' if spans else \'No supported entities detected.\'}\n', 'models.py': '"""GloVe + character CNN recurrent NER and a genuine unconstrained linear-chain CRF."""\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence\n\nclass LinearCRF(nn.Module):\n    """P(y|x), learned start/end and transitions[previous,next]. No hard BIO constraints."""\n    def __init__(self,n):\n        super().__init__();self.start=nn.Parameter(torch.empty(n));self.end=nn.Parameter(torch.empty(n));self.transitions=nn.Parameter(torch.empty(n,n))\n        for p in self.parameters():nn.init.uniform_(p,-.1,.1)\n    def validate(self,emissions,tags,mask):\n        assert emissions.ndim==3 and mask.dtype==torch.bool and mask.shape==emissions.shape[:2]\n        assert mask[:,0].all() and not ((~mask[:,:-1])&mask[:,1:]).any(),\'Masks must be nonempty right padded\'\n        if tags is not None:\n            assert tags.shape==mask.shape and tags.dtype==torch.long\n            assert (tags>=0).all() and (tags<emissions.shape[-1]).all(),\'CRF requires valid tags even on padding; never -100\'\n    def nll(self,emissions,tags,mask):\n        self.validate(emissions,tags,mask)\n        with torch.autocast(device_type=emissions.device.type,enabled=False):\n            e=emissions.float();b=e.shape[0];ix=torch.arange(b,device=e.device)\n            score=self.start[tags[:,0]]+e[ix,0,tags[:,0]]\n            alpha=self.start+e[:,0]\n            for t in range(1,e.shape[1]):\n                score=score+(self.transitions[tags[:,t-1],tags[:,t]]+e[ix,t,tags[:,t]])*mask[:,t]\n                nxt=torch.logsumexp(alpha.unsqueeze(2)+self.transitions.unsqueeze(0),dim=1)+e[:,t]\n                alpha=torch.where(mask[:,t,None],nxt,alpha)\n            last=mask.long().sum(1)-1;score=score+self.end[tags[ix,last]]\n            logz=torch.logsumexp(alpha+self.end,dim=1);loss=(logz-score).mean()\n            assert loss.dtype==torch.float32 and torch.isfinite(loss)\n            return loss\n    @torch.no_grad()\n    def decode(self,emissions,mask):\n        self.validate(emissions,None,mask)\n        with torch.autocast(device_type=emissions.device.type,enabled=False):\n            e=emissions.float();score=self.start+e[:,0];history=[]\n            for t in range(1,e.shape[1]):\n                best,back=(score.unsqueeze(2)+self.transitions.unsqueeze(0)).max(1)\n                score=torch.where(mask[:,t,None],best+e[:,t],score);history.append(back)\n            ends=(score+self.end).argmax(1);paths=[]\n            for b,length in enumerate(mask.sum(1).tolist()):\n                tag=int(ends[b]);path=[tag]\n                for t in range(length-2,-1,-1):tag=int(history[t][b,tag]);path.append(tag)\n                paths.append(path[::-1])\n            return paths\n\nclass RecurrentNER(nn.Module):\n    def __init__(self,cfg,embeddings=None):\n        super().__init__();self.cfg=cfg\n        self.word=nn.Embedding(cfg[\'n_words\'],50,padding_idx=0)\n        if embeddings is not None:self.word.weight.data.copy_(torch.as_tensor(embeddings))\n        self.char=nn.Embedding(cfg[\'n_chars\'],30,padding_idx=0)\n        self.char_conv=nn.Conv1d(30,50,3,padding=1)\n        self.dropout=nn.Dropout(.3)\n        self.encoder=nn.LSTM(100,128,batch_first=True,bidirectional=cfg[\'model\']!=\'lstm\',num_layers=1)\n        self.head=nn.Linear(128*(1 if cfg[\'model\']==\'lstm\' else 2),cfg[\'n_labels\'])\n        self.crf=LinearCRF(cfg[\'n_labels\']) if cfg[\'model\']==\'bilstm_crf\' else None\n    def forward(self,words,chars,mask):\n        b,t,c=chars.shape;flat=chars.reshape(b*t,c);valid=flat!=0\n        encoded=self.char_conv(self.char(flat).transpose(1,2)).relu()\n        encoded=encoded.masked_fill(~valid[:,None,:],-1e4).max(2).values\n        encoded=torch.where(valid.any(1,keepdim=True),encoded,torch.zeros_like(encoded)).reshape(b,t,50)\n        x=self.dropout(torch.cat((self.word(words),encoded),dim=-1))\n        packed=pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)\n        encoded,_=self.encoder(packed);encoded,_=pad_packed_sequence(encoded,batch_first=True,total_length=t)\n        return self.head(self.dropout(encoded))\n    def loss(self,emissions,tags,mask):\n        return self.crf.nll(emissions,tags,mask) if self.crf else F.cross_entropy(emissions[mask].float(),tags[mask])\n    def decode(self,emissions,mask):\n        return self.crf.decode(emissions,mask) if self.crf else [r[:int(n)].tolist() for r,n in zip(emissions.argmax(-1),mask.sum(1))]\n\ndef collate_recurrent(rows,vocab):\n    b=len(rows);t=max(len(r[\'tokens\']) for r in rows)\n    cap=vocab[\'char_max\'];c=max(1,min(cap,max(len(w) for r in rows for w in r[\'tokens\'])))\n    words=torch.zeros(b,t,dtype=torch.long);chars=torch.zeros(b,t,c,dtype=torch.long)\n    mask=torch.zeros(b,t,dtype=torch.bool);tags=torch.full((b,t),vocab[\'labels\'].index(\'O\'),dtype=torch.long)\n    lid={x:i for i,x in enumerate(vocab[\'labels\'])}\n    for i,r in enumerate(rows):\n        for j,w in enumerate(r[\'tokens\']):\n            words[i,j]=vocab[\'words\'].get(w,1);mask[i,j]=True\n            for k,ch in enumerate(w[:c]):chars[i,j,k]=vocab[\'chars\'].get(ch,1)\n            if \'tags\' in r:tags[i,j]=lid[r[\'tags\'][j]]\n    return {\'words\':words,\'chars\':chars,\'mask\':mask,\'tags\':tags}\n\ndef load_recurrent(checkpoint,device=\'cpu\'):\n    obj=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n    model=RecurrentNER(obj[\'config\']);model.load_state_dict(obj[\'state\']);model.to(device).eval()\n    return model\n', 'pipeline.py': '"""Run all stages in fresh processes; the notebook kernel imports no ML libraries."""\nimport os,sys,shutil,subprocess,json\nfrom pathlib import Path\nfrom common import *\nfrom scheduler import train_all,discover_devices,worker_env\n\ndef stage(source,name,args,env=None):\n    env=dict(os.environ if env is None else env);env[\'MPLBACKEND\']=\'Agg\'\n    print(\'\\n=== \'+name+\' ===\',flush=True)\n    subprocess.run([sys.executable,str(source/name),*args],env=env,check=True)\n\ndef preflight_root(root):\n    root=Path(root);check=root/\'preflight\';(check/\'prepared\').mkdir(parents=True,exist_ok=True)\n    for name in [\'train.json\',\'validation.json\',\'vocab.json\',\'embeddings.npy\',\'length_policy.json\',\'identity.json\']:\n        shutil.copy2(root/\'prepared\'/name,check/\'prepared\'/name)\n    dest=check/\'assets\'\n    if not dest.exists():dest.symlink_to((root/\'assets\').resolve(),target_is_directory=True)\n    return check\n\ndef main(root,mode):\n    root=Path(root).resolve();root.mkdir(exist_ok=True,parents=True);source=Path(__file__).parent\n    stage(source,\'prepare.py\',[\'--root\',str(root),\'--mode\',mode])\n    stage(source,\'verify.py\',[\'--root\',str(root)])\n    devices,info=discover_devices()\n    if mode==\'prep\':\n        print(\'Preparation complete; no training requested.\',flush=True);return\n    if mode==\'full\':\n        check=preflight_root(root);train_all(check,\'smoke\')\n        stage(source,\'finalize.py\',[\'--root\',str(check),\'--mode\',\'smoke\'],worker_env(devices[0]))\n        shutil.copytree(check/\'reports\',root/\'reports/preflight\',dirs_exist_ok=True)\n        if devices==[\'cpu\']:\n            raise RuntimeError(\'All bounded preflight checks passed on CPU. Full training requires a Kaggle GPU accelerator; select T4x2 or one GPU and Run All. No factory reset/restart needed.\')\n    train_all(root,mode)\n    stage(source,\'finalize.py\',[\'--root\',str(root),\'--mode\',mode],worker_env(devices[0]))\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--mode\',choices=[\'full\',\'smoke\',\'prep\'],default=\'full\');a=p.parse_args();main(a.root,a.mode)\n', 'prepare.py': '"""Data-only, hash-checked acquisition and full audit. No dataset loading script runs."""\nimport os,json,urllib.request,gzip,shutil,time\nfrom pathlib import Path\nfrom collections import Counter,defaultdict\nimport numpy as np\nimport pyarrow.parquet as pq\nfrom common import *\n\ndef fetch(url,path,sha256=None,md5=None):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    if not path.exists() or (sha256 and file_hash(path)!=sha256):\n        for attempt in range(3):\n            try:\n                print(\'Downloading\',url,flush=True)\n                with urllib.request.urlopen(url,timeout=120) as r,open(str(path)+\'.part\',\'wb\') as f:\n                    shutil.copyfileobj(r,f,1024*1024)\n                Path(str(path)+\'.part\').replace(path);break\n            except Exception:\n                if attempt==2:raise\n                time.sleep(1+attempt)\n    if sha256:assert file_hash(path)==sha256, f\'Checksum mismatch: {path}\'\n    if md5:assert file_hash(path,\'md5\')==md5, f\'Published MD5 mismatch: {path}\'\n    return path\n\ndef summary(values):\n    a=np.asarray(values)\n    return {\'min\':int(a.min()),\'median\':float(np.median(a)),\'p95\':float(np.percentile(a,95)),\n            \'p99\':float(np.percentile(a,99)),\'max\':int(a.max())}\n\ndef metadata_labels(schema):\n    metadata=json.loads(schema.metadata[b\'huggingface\'])\n    f=metadata[\'info\'][\'features\'][\'ner_tags\']\n    while \'names\' not in f:\n        f=f.get(\'feature\',f.get(\'list\',f.get(\'sequence\')))\n        if f is None:raise ValueError(\'Missing NER ClassLabel metadata\')\n    names=f[\'names\']\n    return [names[str(i)] for i in range(len(names))] if isinstance(names,dict) else names\n\ndef canonical_rows(path,split,labels):\n    table=pq.read_table(path); assert set(table.column_names)=={\'id\',\'tokens\',\'pos_tags\',\'chunk_tags\',\'ner_tags\'}\n    assert metadata_labels(table.schema)==labels\n    result=[];ledger=[]\n    for index,row in enumerate(table.to_pylist()):\n        tokens=row[\'tokens\'];ids=row[\'ner_tags\']\n        assert tokens and len(tokens)==len(ids)==len(row[\'pos_tags\'])==len(row[\'chunk_tags\']), (split,index,\'length\')\n        assert all(isinstance(t,str) and t for t in tokens),(split,index,\'empty token\')\n        assert all(isinstance(t,int) and 0<=t<len(labels) for t in ids),(split,index,\'label ID\')\n        original=[labels[t] for t in ids];tags,changes=normalize_iob(original)\n        result.append({\'id\':str(row[\'id\']),\'split\':split,\'index\':index,\'tokens\':tokens,\'original_tags\':original,\'tags\':tags})\n        for c in changes:ledger.append({\'split\':split,\'id\':row[\'id\'],**c})\n    assert len({r[\'id\'] for r in result})==len(result),\'Duplicate sentence IDs\'\n    return result,ledger,str(table.schema)\n\ndef build(root,mode=\'full\'):\n    from transformers import AutoTokenizer\n    root=Path(root);src=root/\'assets\';prep=root/\'prepared\';reports=root/\'reports\'\n    for p in [src,prep,reports]:p.mkdir(parents=True,exist_ok=True)\n    lock=read_json(Path(__file__).with_name(\'source_lock.json\'))\n    downloaded={}\n    for name,info in lock[\'files\'].items():\n        # Test audit only in full/prep. Smoke never downloads or opens test data.\n        if mode==\'smoke\' and name in (\'test.parquet\',\'original_corpus.zip\'):continue\n        if name.startswith(\'distilbert/\') and name.endswith(\'model.safetensors\'):continue # worker fetches pinned weights\n        p=fetch(info[\'url\'],src/name,info[\'sha256\'],info.get(\'published_md5\'))\n        downloaded[name]={\'sha256\':file_hash(p),\'bytes\':p.stat().st_size}\n    tok=AutoTokenizer.from_pretrained(src/\'distilbert\',use_fast=True,local_files_only=True)\n    assert tok.is_fast\n    labels=metadata_labels(pq.read_schema(src/\'train.parquet\'))\n    assert set(labels)=={\'O\'}|{p+\'-\'+t for p in [\'B\',\'I\'] for t in TYPES}\n    rows={};conversions=[];audit={\'source\':lock,\'downloaded\':downloaded,\'labels_by_numeric_id\':labels,\n        \'split_policy\':\'Official boundaries; all rows retained; overlap reported without deduplication.\',\n        \'normalization_rule\':\'Orphan/type-changing I-X -> B-X only; B-X preserved.\',\n        \'predictive_features\':\'words, characters, subwords only; no POS/chunk features\',\n        \'mode\':mode,\'splits\':{},\'exclusion_ledger\':[]}\n    splits=[\'train\',\'validation\']+([\'test\'] if mode!=\'smoke\' else [])\n    for split in splits:\n        rs,ledger,schema=canonical_rows(src/(split+\'.parquet\'),split,labels);rows[split]=rs;conversions+=ledger\n        tags=Counter(t for r in rs for t in r[\'tags\']);ents=Counter();ent_lengths=[]\n        sub=[];unk=0\n        for r in rs:\n            spans=strict_spans(r[\'tags\']);ents.update(t for t,a,b in spans);ent_lengths += [b-a for t,a,b in spans]\n            ids=tok(r[\'tokens\'],is_split_into_words=True,add_special_tokens=True,truncation=False)[\'input_ids\']\n            sub.append(len(ids));unk+=ids.count(tok.unk_token_id)\n        audit[\'splits\'][split]={\'sentences\':len(rs),\'tokens\':sum(len(r[\'tokens\']) for r in rs),\n            \'fingerprint\':digest(rs),\'schema\':schema,\'empty_or_malformed\':0,\'tag_counts\':dict(tags),\n            \'o_prevalence\':tags[\'O\']/sum(tags.values()),\'entities\':dict(ents),\'entity_lengths\':summary(ent_lengths),\n            \'word_lengths\':summary([len(r[\'tokens\']) for r in rs]),\'subword_lengths\':summary(sub),\'unk_subwords\':unk,\n            \'normalization_conversions\':len(ledger)}\n        write_json(prep/(split+\'.json\'),rs)\n    # Independently compare every token and raw NER label to the upstream loader\'s ZIP as data.\n    if mode!=\'smoke\':\n        import zipfile\n        comparison={}\n        with zipfile.ZipFile(src/\'original_corpus.zip\') as z:\n            for split,name in [(\'train\',\'train.txt\'),(\'validation\',\'valid.txt\'),(\'test\',\'test.txt\')]:\n                original=[];words=[];tags=[]\n                for line in z.read(name).decode(\'utf8\').splitlines()+[\'\']:\n                    if not line or line.startswith(\'-DOCSTART-\'):\n                        if words:original.append((words,tags));words=[];tags=[]\n                    else:\n                        fields=line.split(\' \');assert len(fields)==4\n                        words.append(fields[0]);tags.append(fields[3])\n                assert original==[(r[\'tokens\'],r[\'original_tags\']) for r in rows[split]], \'Mirror split differs from original source\'\n                comparison[split]={\'matched_all_tokens_and_raw_ner_labels\':True,\'sentences\':len(original)}\n        write_json(reports/\'provenance_comparison.json\',comparison)\n    train_words=sorted({t for r in rows[\'train\'] for t in r[\'tokens\']})\n    train_chars=sorted({c for t in train_words for c in t})\n    vocab={\'words\':{\'<PAD>\':0,\'<UNK>\':1,**{w:i+2 for i,w in enumerate(train_words)}},\n           \'chars\':{\'<PAD>\':0,\'<UNK>\':1,**{c:i+2 for i,c in enumerate(train_chars)}},\'labels\':labels,\n           \'char_max\':max(map(len,train_words)), \'train_only\':True}\n    assert len(vocab[\'words\'])==len(train_words)+2\n    # Word and character lengths never use validation to change vocab or char cap.\n    maximum=audit[\'splits\'][\'train\'][\'subword_lengths\'][\'max\']\n    max_length=256 if maximum<=256 else min(512,maximum)\n    length={\'max_subwords\':int(max_length),\'word_policy\':\'No recurrent word truncation; packed dynamic batches\',\n            \'char_max\':vocab[\'char_max\'],\'char_policy\':\'Train max; held-out/app longer chars clipped, count reported\',\n            \'transformer_policy\':\'Greedy whole-word chunks; no overlap; label-independent; reconstruct before scoring\',\n            \'app_recurrent_chunk_words\':256,\'app_max_characters\':20000,\'app_sentence_policy\':APP_TOKENIZER_POLICY}\n    for split,rs in rows.items():\n        feats,mapping=aligned_features(rs,tok,labels,max_length)\n        crossed=0\n        by_sentence=defaultdict(list)\n        for m in mapping:by_sentence[m[\'sentence\']].append(m[\'b\'])\n        for sid,r in enumerate(rs):\n            cuts=set(by_sentence[sid][:-1]);crossed+=sum(any(a<c<b for c in cuts) for typ,a,b in strict_spans(r[\'tags\']))\n        oov=sum(w not in vocab[\'words\'] for r in rs for w in r[\'tokens\'])\n        chars=[c for r in rs for w in r[\'tokens\'] for c in w]\n        audit[\'splits\'][split].update({\'word_oov_tokens\':oov,\'word_oov_rate\':oov/audit[\'splits\'][split][\'tokens\'],\n            \'char_oov\':sum(c not in vocab[\'chars\'] for c in chars),\'char_oov_rate\':sum(c not in vocab[\'chars\'] for c in chars)/len(chars),\n            \'words_over_char_cap\':sum(len(w)>vocab[\'char_max\'] for r in rs for w in r[\'tokens\']),\n            \'transformer_chunks\':len(feats),\'chunk_crossed_gold_entities\':crossed,\'aligned_words\':sum(len(m[\'first\']) for m in mapping)})\n        assert audit[\'splits\'][split][\'aligned_words\']==audit[\'splits\'][split][\'tokens\']\n    counts={s:Counter(tuple(r[\'tokens\']) for r in rs) for s,rs in rows.items()}\n    duplicate={\'within\':{},\'across\':{}}\n    for s,c in counts.items():\n        variants=defaultdict(set)\n        for r in rows[s]:variants[tuple(r[\'tokens\'])].add(tuple(r[\'tags\']))\n        differing=[{\'tokens\':list(k),\'tag_variants\':sorted(v)} for k,v in variants.items() if len(v)>1]\n        duplicate[\'within\'][s]={\'repeated_keys\':sum(n>1 for n in c.values()),\'extra_rows\':sum(n-1 for n in c.values()),\'annotation_differences\':differing}\n    for i,a in enumerate(splits):\n        for b in splits[i+1:]:\n            common=set(counts[a])&set(counts[b]); diffs=[]\n            for tokens in sorted(common):\n                aa={tuple(r[\'tags\']) for r in rows[a] if tuple(r[\'tokens\'])==tokens}\n                bb={tuple(r[\'tags\']) for r in rows[b] if tuple(r[\'tokens\'])==tokens}\n                if aa!=bb:diffs.append({\'tokens\':tokens,a:sorted(aa),b:sorted(bb)})\n            duplicate[\'across\'][a+\'__\'+b]={\'sentence_keys\':len(common),\'rows_a\':sum(counts[a][x] for x in common),\n                \'rows_b\':sum(counts[b][x] for x in common),\'annotation_differences\':diffs}\n    # Retain only training word vectors. A set of vocabulary strings supports descriptive held-out coverage.\n    matrix=np.random.default_rng(42).normal(0,.05,(len(vocab[\'words\']),50)).astype(\'float32\');matrix[0]=0\n    needed=defaultdict(list)\n    for w,i in vocab[\'words\'].items():\n        if i>=2:needed[w.lower()].append(i)\n    found=set();all_glove=set();bad=0\n    with gzip.open(src/\'glove-wiki-gigaword-50.gz\',\'rt\',encoding=\'utf8\') as f:\n        total,dim=map(int,f.readline().split());assert (total,dim)==(400000,50)\n        count=0\n        for line in f:\n            bits=line.rstrip().split(\' \');word=bits[0]; count+=1\n            if len(bits)!=51:raise ValueError(\'Malformed GloVe row\')\n            all_glove.add(word)\n            if word in needed:\n                vec=np.asarray(bits[1:],dtype=np.float32);assert np.isfinite(vec).all()\n                for i in needed[word]:matrix[i]=vec;found.add(i)\n        assert count==total\n    assert found,\'Pretrained embedding failed: refusing all-random word layer\'\n    np.save(prep/\'embeddings.npy\',matrix)\n    for split,rs in rows.items():\n        words=[w for r in rs for w in r[\'tokens\']]\n        audit[\'splits\'][split][\'external_glove_token_coverage\']=sum(w.lower() in all_glove for w in words)/len(words)\n        audit[\'splits\'][split][\'usable_task_glove_token_coverage\']=sum(vocab[\'words\'].get(w,1) in found for w in words)/len(words)\n    audit[\'embedding\']={\'source\':lock[\'embedding_metadata\'],\'train_vocab_rows\':len(train_words),\n        \'pretrained_rows\':len(found),\'type_coverage\':len(found)/len(train_words),\n        \'matrix_shape\':list(matrix.shape),\'matrix_sha256\':file_hash(prep/\'embeddings.npy\'),\n        \'oov_policy\':\'One reproducibly initialized UNK row plus learned characters; known train misses initialized by seed42\',\n        \'fine_tunable\':True,\'pad_fixed_zero\':True}\n    audit[\'duplicates\']=duplicate;audit[\'length_policy\']=length\n    audit[\'observed_iob\']=\'IOB2\' if not conversions else \'IOB1-compatible orphan I observed; normalized with ledger\'\n    audit[\'conversion_count\']=len(conversions)\n    write_json(reports/\'normalization_ledger.json\',conversions)\n    # Corpus provides tokens rather than original article character offsets; visualization uses explicit space-joined display only.\n    import html\n    colors={\'PER\':\'#c7d2fe\',\'ORG\':\'#fde68a\',\'LOC\':\'#a7f3d0\',\'MISC\':\'#fecdd3\'}\n    snippets=[]\n    examples=[]\n    for r in rows[\'train\'][:8]:\n        chunks=[];last=0\n        for typ,a,b in strict_spans(r[\'tags\']):\n            chunks.append(html.escape(\' \'.join(r[\'tokens\'][last:a])))\n            chunks.append(\'<mark style="background:\'+colors[typ]+\'">\'+html.escape(\' \'.join(r[\'tokens\'][a:b]))+\' [\'+typ+\']</mark>\');last=b\n        chunks.append(html.escape(\' \'.join(r[\'tokens\'][last:])))\n        snippets.append(\'<p>\'+str(r[\'id\'])+\': \'+\' \'.join(chunks)+\'</p>\');examples.append(r)\n    write_json(reports/\'annotated_train_examples.json\',examples)\n    reports.joinpath(\'annotated_train_examples.html\').write_text(\'<!doctype html><meta charset="utf-8"><h1>Gold train examples</h1><p>Space-joined corpus tokens for display; original article offsets are not supplied by this dataset.</p>\'+\'\'.join(snippets))\n    write_json(prep/\'vocab.json\',vocab);write_json(prep/\'length_policy.json\',length);write_json(reports/\'data_audit.json\',audit)\n    identity={\'split_fingerprints\':{s:audit[\'splits\'][s][\'fingerprint\'] for s in splits},\n              \'vocab\':digest(vocab),\'embedding\':audit[\'embedding\'][\'matrix_sha256\'],\'length_policy\':length}\n    write_json(prep/\'identity.json\',identity)\n    reference=read_json(Path(__file__).with_name(\'reference.json\'))\n    checks={}\n    if reference.get(\'observed\',False):\n        for s in splits:\n            expected=reference[\'splits\'][s]\n            for key in [\'sentences\',\'tokens\',\'fingerprint\',\'normalization_conversions\',\'entities\',\'tag_counts\',\'subword_lengths\',\'aligned_words\']:\n                assert audit[\'splits\'][s][key]==expected[key],f\'Reference mismatch: {s}/{key}\'\n            checks[s]=\'exact counts, tags, spans, lengths and canonical fingerprint matched\'\n        assert identity[\'vocab\']==reference[\'vocab_fingerprint\']\n        assert identity[\'embedding\']==reference[\'embedding_matrix_sha256\']\n    write_json(reports/\'reference_comparison.json\',{\'cloud_observed_reference\':reference.get(\'observed\',False),\'checks\':checks})\n    make_figures(audit,reports)\n    print(json.dumps({s:{k:v for k,v in info.items() if k in [\'sentences\',\'tokens\',\'entities\',\'subword_lengths\',\'word_oov_rate\']} for s,info in audit[\'splits\'].items()},indent=2),flush=True)\n    return audit\n\ndef make_figures(audit,reports):\n    import os;os.environ[\'MPLBACKEND\']=\'Agg\'\n    import matplotlib;matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,ax=plt.subplots(1,3,figsize=(14,4))\n    ss=list(audit[\'splits\'])\n    ax[0].bar(ss,[audit[\'splits\'][s][\'tokens\'] for s in ss]);ax[0].set_title(\'Official split token counts\')\n    for j,t in enumerate(TYPES):ax[1].bar(np.arange(len(ss))+.18*j,[audit[\'splits\'][s][\'entities\'].get(t,0) for s in ss],.18,label=t)\n    ax[1].set_xticks(np.arange(len(ss))+.27,ss);ax[1].legend();ax[1].set_title(\'Gold exact entities\')\n    ax[2].bar(ss,[audit[\'splits\'][s][\'word_oov_rate\'] for s in ss]);ax[2].set_title(\'Train-vocabulary word OOV rate\')\n    fig.tight_layout();fig.savefig(reports/\'eda.png\',dpi=130);plt.close(fig)\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--mode\',choices=[\'full\',\'smoke\',\'prep\'],default=\'full\');a=p.parse_args();build(a.root,a.mode)\n', 'reference.json': '{\n  "observed": true,\n  "splits": {\n    "train": {\n      "sentences": 14041,\n      "tokens": 203621,\n      "fingerprint": "94c7192df8c4d806d437e87df07b8cf831b97af9fd22922671f7146775d90642",\n      "schema": "id: string\\ntokens: list<element: string>\\n  child 0, element: string\\npos_tags: list<element: int64>\\n  child 0, element: int64\\nchunk_tags: list<element: int64>\\n  child 0, element: int64\\nner_tags: list<element: int64>\\n  child 0, element: int64\\n-- schema metadata --\\nhuggingface: \'{\\"info\\": {\\"features\\": {\\"id\\": {\\"dtype\\": \\"string\\", \\"_type\\": \\"\' + 914",\n      "empty_or_malformed": 0,\n      "tag_counts": {\n        "B-ORG": 6321,\n        "O": 169578,\n        "B-MISC": 3438,\n        "B-PER": 6600,\n        "I-PER": 4528,\n        "B-LOC": 7140,\n        "I-ORG": 3704,\n        "I-MISC": 1155,\n        "I-LOC": 1157\n      },\n      "o_prevalence": 0.832811939829389,\n      "entities": {\n        "ORG": 6321,\n        "MISC": 3438,\n        "PER": 6600,\n        "LOC": 7140\n      },\n      "entity_lengths": {\n        "min": 1,\n        "median": 1.0,\n        "p95": 3.0,\n        "p99": 4.0,\n        "max": 10\n      },\n      "word_lengths": {\n        "min": 1,\n        "median": 10.0,\n        "p95": 37.0,\n        "p99": 46.0,\n        "max": 113\n      },\n      "subword_lengths": {\n        "min": 3,\n        "median": 17.0,\n        "p95": 47.0,\n        "p99": 58.0,\n        "max": 173\n      },\n      "unk_subwords": 0,\n      "normalization_conversions": 0,\n      "word_oov_tokens": 0,\n      "word_oov_rate": 0.0,\n      "char_oov": 0,\n      "char_oov_rate": 0.0,\n      "words_over_char_cap": 0,\n      "transformer_chunks": 14041,\n      "chunk_crossed_gold_entities": 0,\n      "aligned_words": 203621,\n      "external_glove_token_coverage": 0.9772518551622867,\n      "usable_task_glove_token_coverage": 0.9772518551622867\n    },\n    "validation": {\n      "sentences": 3250,\n      "tokens": 51362,\n      "fingerprint": "c25ae5cd173d132b8136e41fc0cc6a6b78c4fabbcaed58aae2d0e08a4c60e8bc",\n      "schema": "id: string\\ntokens: list<element: string>\\n  child 0, element: string\\npos_tags: list<element: int64>\\n  child 0, element: int64\\nchunk_tags: list<element: int64>\\n  child 0, element: int64\\nner_tags: list<element: int64>\\n  child 0, element: int64\\n-- schema metadata --\\nhuggingface: \'{\\"info\\": {\\"features\\": {\\"id\\": {\\"dtype\\": \\"string\\", \\"_type\\": \\"\' + 914",\n      "empty_or_malformed": 0,\n      "tag_counts": {\n        "O": 42759,\n        "B-ORG": 1341,\n        "B-LOC": 1837,\n        "B-MISC": 922,\n        "I-MISC": 346,\n        "B-PER": 1842,\n        "I-PER": 1307,\n        "I-LOC": 257,\n        "I-ORG": 751\n      },\n      "o_prevalence": 0.8325026284023208,\n      "entities": {\n        "ORG": 1341,\n        "LOC": 1837,\n        "MISC": 922,\n        "PER": 1842\n      },\n      "entity_lengths": {\n        "min": 1,\n        "median": 1.0,\n        "p95": 3.0,\n        "p99": 4.0,\n        "max": 10\n      },\n      "word_lengths": {\n        "min": 1,\n        "median": 11.0,\n        "p95": 39.0,\n        "p99": 49.0,\n        "max": 109\n      },\n      "subword_lengths": {\n        "min": 3,\n        "median": 18.0,\n        "p95": 49.0,\n        "p99": 62.0,\n        "max": 151\n      },\n      "unk_subwords": 0,\n      "normalization_conversions": 0,\n      "word_oov_tokens": 4293,\n      "word_oov_rate": 0.08358319380086446,\n      "char_oov": 0,\n      "char_oov_rate": 0.0,\n      "words_over_char_cap": 0,\n      "transformer_chunks": 3250,\n      "chunk_crossed_gold_entities": 0,\n      "aligned_words": 51362,\n      "external_glove_token_coverage": 0.9794789922510806,\n      "usable_task_glove_token_coverage": 0.9097971262801293\n    },\n    "test": {\n      "sentences": 3453,\n      "tokens": 46435,\n      "fingerprint": "141b3a4f1b1da2f59ce0a979e43ef44800bbb3732ee641c0f029a8e8179dc3e2",\n      "schema": "id: string\\ntokens: list<element: string>\\n  child 0, element: string\\npos_tags: list<element: int64>\\n  child 0, element: int64\\nchunk_tags: list<element: int64>\\n  child 0, element: int64\\nner_tags: list<element: int64>\\n  child 0, element: int64\\n-- schema metadata --\\nhuggingface: \'{\\"info\\": {\\"features\\": {\\"id\\": {\\"dtype\\": \\"string\\", \\"_type\\": \\"\' + 914",\n      "empty_or_malformed": 0,\n      "tag_counts": {\n        "O": 38323,\n        "B-LOC": 1668,\n        "B-PER": 1617,\n        "I-PER": 1156,\n        "I-LOC": 257,\n        "B-MISC": 702,\n        "I-MISC": 216,\n        "B-ORG": 1661,\n        "I-ORG": 835\n      },\n      "o_prevalence": 0.8253041886508022,\n      "entities": {\n        "LOC": 1668,\n        "PER": 1617,\n        "MISC": 702,\n        "ORG": 1661\n      },\n      "entity_lengths": {\n        "min": 1,\n        "median": 1.0,\n        "p95": 3.0,\n        "p99": 4.0,\n        "max": 6\n      },\n      "word_lengths": {\n        "min": 1,\n        "median": 9.0,\n        "p95": 37.0,\n        "p99": 47.0,\n        "max": 124\n      },\n      "subword_lengths": {\n        "min": 3,\n        "median": 16.0,\n        "p95": 46.0,\n        "p99": 59.0,\n        "max": 148\n      },\n      "unk_subwords": 0,\n      "normalization_conversions": 0,\n      "word_oov_tokens": 5656,\n      "word_oov_rate": 0.1218046731990955,\n      "char_oov": 1,\n      "char_oov_rate": 4.997201567122411e-06,\n      "words_over_char_cap": 0,\n      "transformer_chunks": 3453,\n      "chunk_crossed_gold_entities": 0,\n      "aligned_words": 46435,\n      "external_glove_token_coverage": 0.9713362765155594,\n      "usable_task_glove_token_coverage": 0.8741897275761817\n    }\n  },\n  "vocab_fingerprint": "3a0eddc128ba9bc9e6ab7607ac620d3141f14a4475abd656507687ec8fbd16fa",\n  "embedding_matrix_sha256": "0df3b88b35ca0f93909ebc552c946f25aaa6980d1b59661e68370f89ccafaaa9"\n}', 'requirements.txt': 'transformers==4.57.1\naccelerate==1.11.0\nhuggingface-hub==0.36.0\ntokenizers==0.22.1\nsafetensors==0.6.2\nseqeval==1.2.2\nnumpy==2.2.6\nscipy==1.15.3\nscikit-learn==1.7.2\npyarrow==19.0.1\npandas==2.2.3\nmatplotlib==3.10.3\ngradio==5.49.1\npackaging==25.0\ntqdm==4.67.1\nsocksio==1.0.0\npydantic==2.11.10\nfastapi==0.162.1\nstarlette==0.52.1\nhttpx==0.28.1\ngradio-client==1.13.3\nuvicorn==0.54.0\nwrapt==1.17.3\n', 'scheduler.py': '"""No torch import in parent. GPU visibility is fixed in each subprocess environment."""\nimport os,sys,json,time,subprocess,signal,shutil\nfrom pathlib import Path\nfrom common import read_json,write_json,digest,file_hash\nMODELS=[\'distilbert\',\'lstm\',\'bilstm\',\'bilstm_crf\']\nDISTRIBUTED_KEYS=[\'LOCAL_RANK\',\'RANK\',\'WORLD_SIZE\',\'LOCAL_WORLD_SIZE\',\'MASTER_ADDR\',\'MASTER_PORT\',\'ACCELERATE_USE_CPU\',\'ACCELERATE_MIXED_PRECISION\']\ndef worker_env(device):\n    env=os.environ.copy()\n    for k in DISTRIBUTED_KEYS:env.pop(k,None)\n    env.update({\'CUDA_VISIBLE_DEVICES\':str(device) if device!=\'cpu\' else \'\',\n        \'OMP_NUM_THREADS\':\'2\',\'MKL_NUM_THREADS\':\'2\',\'OPENBLAS_NUM_THREADS\':\'2\',\n        \'TOKENIZERS_PARALLELISM\':\'false\',\'PYTHONUNBUFFERED\':\'1\',\'WANDB_DISABLED\':\'true\',\'HF_HUB_DISABLE_TELEMETRY\':\'1\'})\n    return env\n\ndef discover_devices(python=sys.executable):\n    cmd=[python,\'-c\',\'import torch,json; print(json.dumps({"count":torch.cuda.device_count(),"torch":torch.__version__,"cuda":torch.version.cuda}))\']\n    env=os.environ.copy()\n    for k in DISTRIBUTED_KEYS:env.pop(k,None)\n    info=json.loads(subprocess.check_output(cmd,env=env,text=True).strip().splitlines()[-1])\n    n=info[\'count\'];existing=os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n    devices=existing.split(\',\')[:n] if existing else [str(i) for i in range(n)]\n    return devices[:2] or [\'cpu\'],info\n\ndef completed(root,model,identity):\n    p=Path(root)/\'models\'/model/\'done.json\'\n    if not p.exists():return False\n    done=read_json(p)\n    if done[\'experiment_id\']!=identity:raise RuntimeError(f\'Incompatible completed model {model}; choose a new RUN_NAME, do not mix results\')\n    for f,h in done[\'files\'].items():\n        q=p.parent/f\n        if not q.is_file() or file_hash(q)!=h:raise RuntimeError(f\'Corrupt completed artifact: {q}\')\n    return True\n\ndef stop_active(active,events,reason):\n    for p,job,dev,f in active:\n        if p.poll() is None:\n            events.append({\'event\':\'cancel\',\'model\':job,\'device\':dev,\'time\':time.time(),\'reason\':reason,\'pid\':p.pid})\n            try:os.killpg(p.pid,signal.SIGTERM)\n            except ProcessLookupError:pass\n    deadline=time.time()+5\n    for p,job,dev,f in active:\n        try:p.wait(timeout=max(.01,deadline-time.time()))\n        except subprocess.TimeoutExpired:\n            try:os.killpg(p.pid,signal.SIGKILL)\n            except ProcessLookupError:pass\n            p.wait()\n        f.close()\n\ndef run_queue(jobs,devices,command,root,env_factory=worker_env,poll=.5):\n    """One job per device; first failure/interrupt cancels every active process group."""\n    root=Path(root);logs=root/\'logs\';logs.mkdir(parents=True,exist_ok=True)\n    events=[];queue=list(jobs);active=[];last_echo=0;positions={}\n    try:\n        while queue or active:\n            used={d for p,j,d,f in active}\n            for dev in devices:\n                if queue and dev not in used:\n                    job=queue.pop(0);f=open(logs/(job+\'.log\'),\'w\')\n                    p=subprocess.Popen(command(job,dev),env=env_factory(dev),stdout=f,stderr=subprocess.STDOUT,start_new_session=True)\n                    active.append((p,job,dev,f));used.add(dev)\n                    events.append({\'event\':\'start\',\'model\':job,\'device\':dev,\'pid\':p.pid,\'time\':time.time()})\n                    print(f\'Started {job} on {dev}; log: {logs/(job+".log")}\',flush=True)\n            finished=[]\n            for item in active:\n                p,job,dev,f=item;code=p.poll()\n                if code is not None:\n                    events.append({\'event\':\'exit\',\'model\':job,\'device\':dev,\'pid\':p.pid,\'code\':code,\'time\':time.time()})\n                    if code:\n                        print((logs/(job+\'.log\')).read_text(errors=\'replace\')[-12000:],flush=True)\n                        raise RuntimeError(f\'{job} exited {code}; all active jobs cancelled. Completed compatible models remain reusable.\')\n                    f.close();finished.append(item)\n            active=[x for x in active if x not in finished]\n            if time.time()-last_echo>15:\n                for p,j,d,f in active:\n                    data=(logs/(j+\'.log\')).read_text(errors=\'replace\');old=positions.get(j,0)\n                    if len(data)>old:print(f\'[{j} / {d}] \'+data[old:][-2500:],flush=True);positions[j]=len(data)\n                last_echo=time.time()\n            if queue or active:time.sleep(poll)\n    except BaseException as exc:\n        stop_active(active,events,type(exc).__name__);raise\n    finally:write_json(root/\'schedule_events.json\',events)\n    return events\n\ndef experiment_identity(root,mode):\n    import importlib.metadata as im\n    code={p.name:file_hash(p) for p in sorted(Path(__file__).parent.glob(\'*\')) if p.is_file() and p.suffix in (\'.py\',\'.json\',\'.txt\')}\n    versions={name:im.version(name) for name in [\'torch\',\'transformers\',\'accelerate\',\'seqeval\',\'numpy\',\'pyarrow\',\'tokenizers\',\'safetensors\']}\n    config={\'seed\':42,\'rnn_epochs\':15,\'rnn_patience\':3,\'rnn_batch\':32,\'rnn_lr\':.001,\'hidden\':128,\'char_dim\':30,\'char_features\':50,\n            \'dropout\':.3,\'transformer_epochs\':5,\'transformer_patience\':2,\'transformer_batch\':16,\'accumulation\':1,\'transformer_lr\':2e-5,\n            \'mode\':mode,\'data\':read_json(Path(root)/\'prepared/identity.json\'),\'code\':code,\'versions\':versions}\n    return digest(config),config\n\ndef train_all(root,mode=\'full\',force_devices=None):\n    root=Path(root);devices,hardware=discover_devices()\n    if force_devices is not None:devices=force_devices\n    identity,config=experiment_identity(root,mode)\n    write_json(root/\'experiment.json\',{\'id\':identity,\'configuration\':config,\'hardware\':hardware,\'devices\':devices})\n    jobs=[m for m in MODELS if not completed(root,m,identity)]\n    write_json(root/\'reuse_verification.json\',{\'experiment_id\':identity,\'reused\':[m for m in MODELS if m not in jobs],\'scheduled\':jobs,\'time\':time.time()})\n    for m in jobs:\n        p=root/\'models\'/m\n        # Incomplete models restart from pretrained/initial state, never optimizer-resume.\n        if p.exists():shutil.rmtree(p)\n    def command(job,dev):return [sys.executable,str(Path(__file__).with_name(\'worker.py\')),\'--root\',str(root.resolve()),\'--model\',job,\'--mode\',mode,\'--identity\',identity]\n    if jobs:run_queue(jobs,devices,command,root)\n    else:print(\'Reused all four completed models after identity and file-hash verification; no optimizer resume.\',flush=True)\n    for m in MODELS:assert completed(root,m,identity),f\'Missing completion: {m}\'\n    return identity\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--mode\',default=\'full\');a=p.parse_args();train_all(a.root,a.mode)\n', 'source_lock.json': '{\n  "dataset_repository": "BramVanroy/conll2003",\n  "dataset_revision": "4ffbd53d9e0b92b473b9b7dcff12f53e7c17ce0c",\n  "original_repository": "eriktks/conll2003",\n  "original_revision": "3f1cce917ab38486481b062921eb137e7bd3c205",\n  "model_repository": "distilbert/distilbert-base-cased",\n  "model_revision": "6ea81172465e8b0ad3fddeed32b986cdcdcffcf0",\n  "embedding_metadata": {\n    "num_records": 400000,\n    "file_size": 69182535,\n    "base_dataset": "Wikipedia 2014 + Gigaword 5 (6B tokens, uncased)",\n    "reader_code": "https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-50/__init__.py",\n    "license": "http://opendatacommons.org/licenses/pddl/",\n    "parameters": {\n      "dimension": 50\n    },\n    "description": "Pre-trained vectors based on Wikipedia 2014 + Gigaword, 5.6B tokens, 400K vocab, uncased (https://nlp.stanford.edu/projects/glove/).",\n    "preprocessing": "Converted to w2v format with `python -m gensim.scripts.glove2word2vec -i <fname> -o glove-wiki-gigaword-50.txt`.",\n    "read_more": [\n      "https://nlp.stanford.edu/projects/glove/",\n      "https://nlp.stanford.edu/pubs/glove.pdf"\n    ],\n    "checksum": "c289bc5d7f2f02c6dc9f2f9b67641813",\n    "file_name": "glove-wiki-gigaword-50.gz",\n    "parts": 1\n  },\n  "files": {\n    "mirror_README.md": {\n      "url": "https://huggingface.co/datasets/BramVanroy/conll2003/resolve/4ffbd53d9e0b92b473b9b7dcff12f53e7c17ce0c/README.md",\n      "sha256": "957bef95b2fb4db76457e92f3c37717686b79c982d283369b05a5b26a0a286b5",\n      "bytes": 12747\n    },\n    "original_README.md": {\n      "url": "https://huggingface.co/datasets/eriktks/conll2003/resolve/3f1cce917ab38486481b062921eb137e7bd3c205/README.md",\n      "sha256": "d084dc7e14f5537b04a63d685d212db05ed49e9a4cef4ae20264ad0bee777117",\n      "bytes": 12330\n    },\n    "original_script.py": {\n      "url": "https://huggingface.co/datasets/eriktks/conll2003/resolve/3f1cce917ab38486481b062921eb137e7bd3c205/conll2003.py",\n      "sha256": "bf7c353c1dfc5526e2a1ce2a7e27da69eb0d62db760c9f670ea0312bcfbaf77e",\n      "bytes": 9570\n    },\n    "glove-wiki-gigaword-50.gz": {\n      "url": "https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-50/glove-wiki-gigaword-50.gz",\n      "sha256": "5c55f98957aa9fed8d2ac5fb1dcff57af3b23c5a3ee7af3f7945f8d49198eb24",\n      "bytes": 69182535,\n      "published_md5": "c289bc5d7f2f02c6dc9f2f9b67641813"\n    },\n    "train.parquet": {\n      "url": "https://huggingface.co/datasets/BramVanroy/conll2003/resolve/4ffbd53d9e0b92b473b9b7dcff12f53e7c17ce0c/data/train-00000-of-00001.parquet",\n      "sha256": "a92ca6a5a082770a072822441917619e46caa5be2af795273640d972136a7935",\n      "bytes": 1221000\n    },\n    "validation.parquet": {\n      "url": "https://huggingface.co/datasets/BramVanroy/conll2003/resolve/4ffbd53d9e0b92b473b9b7dcff12f53e7c17ce0c/data/validation-00000-of-00001.parquet",\n      "sha256": "78052bbaec76513a03704011bf42e2e1ba5f46fa2d032d40b5039638555b61d2",\n      "bytes": 310076\n    },\n    "test.parquet": {\n      "url": "https://huggingface.co/datasets/BramVanroy/conll2003/resolve/4ffbd53d9e0b92b473b9b7dcff12f53e7c17ce0c/data/test-00000-of-00001.parquet",\n      "sha256": "696edfc316c682fb3a51bf833968922301409d7f452bec044934fed8bbe612f5",\n      "bytes": 281499\n    },\n    "distilbert/config.json": {\n      "url": "https://huggingface.co/distilbert/distilbert-base-cased/resolve/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0/config.json",\n      "sha256": "165376e2c837d836c75ae30fde8c5569beb3ed7325b7473c18a76e4608cdec22",\n      "bytes": 465\n    },\n    "distilbert/tokenizer.json": {\n      "url": "https://huggingface.co/distilbert/distilbert-base-cased/resolve/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0/tokenizer.json",\n      "sha256": "5b3360be30cdf39081153cf9d30bfb0503c011462fc9f77f097127c83ef0f4d9",\n      "bytes": 435797\n    },\n    "distilbert/tokenizer_config.json": {\n      "url": "https://huggingface.co/distilbert/distilbert-base-cased/resolve/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0/tokenizer_config.json",\n      "sha256": "0f6d13e6f4da6f9e24f22ada6bc3be571123d858d7c0c05a8a7cd55a9c23c2e8",\n      "bytes": 49\n    },\n    "distilbert/vocab.txt": {\n      "url": "https://huggingface.co/distilbert/distilbert-base-cased/resolve/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0/vocab.txt",\n      "sha256": "eeaa9875b23b04b4c54ef759d03db9d1ba1554838f8fb26c5d96fa551df93d02",\n      "bytes": 213450\n    },\n    "distilbert/model.safetensors": {\n      "url": "https://huggingface.co/distilbert/distilbert-base-cased/resolve/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0/model.safetensors",\n      "sha256": "64080c2b1998c3bd1214311ab9f1e97b40052fa81d508b8b534a936c4ba1626b",\n      "bytes": 263260784\n    },\n    "original_corpus.zip": {\n      "url": "https://data.deepai.org/conll2003.zip",\n      "sha256": "96a104d174ddae7558bab603f19382c5fe02ff1da5c077a7f3ce2ced1578a2c3",\n      "bytes": 982975\n    }\n  },\n  "provenance": "Mirror card identifies duplicate of eriktks/conll2003. Original loader inspected as text, never executed. ClassLabel metadata from actual Parquet. Source corpus Reuters CoNLL2003 English; license other; consult cards and corpus terms.",\n  "original_corpus_comparison": {\n    "train": {\n      "matched_all_original_tokens_and_raw_tags": true,\n      "sentences": 14041,\n      "tokens": 203621,\n      "raw_file_sha256": "213953b5522fb7924802d883e50f027b2c6ffb5d4d0e63368ed82909f9b60423"\n    },\n    "validation": {\n      "matched_all_original_tokens_and_raw_tags": true,\n      "sentences": 3250,\n      "tokens": 51362,\n      "raw_file_sha256": "c02eb19809eafb83998326a0481942eb37d96f31826ae387cfe8773699969f5b"\n    },\n    "test": {\n      "matched_all_original_tokens_and_raw_tags": true,\n      "sentences": 3453,\n      "tokens": 46435,\n      "raw_file_sha256": "dc4ac7f3d6d01a6bec59261814561da3b287b6b493b97d1f69c580b52ec1831b"\n    }\n  }\n}', 'verify.py': '"""Bounded correctness tests. No real test split is loaded in this module."""\nimport os,sys,time,itertools,tempfile,subprocess,json,math\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom common import *\nfrom models import *\n\ndef run_tests(root):\n    from transformers import AutoTokenizer,DataCollatorForTokenClassification,Trainer,TrainingArguments\n    from seqeval.metrics import f1_score\n    from seqeval.scheme import IOB2\n    root=Path(root);torch.set_num_threads(2);torch.manual_seed(7);evidence={}\n    labels=read_json(root/\'prepared/vocab.json\')[\'labels\'];vocab=read_json(root/\'prepared/vocab.json\')\n    assert normalize_iob([\'I-PER\',\'I-PER\',\'I-ORG\',\'B-ORG\',\'O\',\'I-LOC\'])[0]==[\'B-PER\',\'I-PER\',\'B-ORG\',\'B-ORG\',\'O\',\'B-LOC\']\n    assert len(normalize_iob([\'I-PER\',\'I-PER\',\'I-ORG\',\'B-ORG\',\'O\',\'I-LOC\'])[1])==3\n    try:normalize_iob([\'B-DATE\']);raise AssertionError(\'Bad label accepted\')\n    except ValueError:pass\n    evidence[\'iob_normalization\']=\'orphan and type-change conversion; valid B retained; invalid taxonomy rejected\'\n    raw=\'Dr. Smith met U.S. officials and Jean-Claude in Baden-Wuerttemberg.\'\n    parts=text_sentences(raw)\n    assert len(parts)==1 and [w[0] for w in parts[0]][:4]==[\'Dr.\',\'Smith\',\'met\',\'U.S.\']\n    assert \'Jean-Claude\' in [w[0] for w in parts[0]]\n    raw=\'\\ufeffAlice\\u200bSmith works at Micro\\u00adsoft.\'\n    words=text_sentences(raw)[0]\n    assert words[0][0]==\'AliceSmith\' and words[3][0]==\'Microsoft\'\n    assert highlight_spans(raw,words,[\'B-PER\',\'O\',\'O\',\'B-ORG\',\'O\'])[0][\'text\']==\'Alice\\u200bSmith\'\n    evidence[\'raw_text_v2\']=\'acronyms, titles, hyphens, invisible characters and exact source offsets\'\n\n    g=[[\'B-PER\',\'I-PER\',\'O\',\'B-LOC\',\'B-LOC\']]\n    assert metrics(g,g)[\'f1\']==1.\n    assert metrics([[\'B-PER\',\'I-PER\']],[[\'B-PER\',\'O\']])[\'f1\']==0\n    assert metrics([[\'B-PER\']],[[\'B-ORG\']])[\'f1\']==0\n    assert metrics([[\'B-PER\']],[[\'O\']])[\'f1\']==0\n    assert metrics([[\'B-PER\',\'I-PER\']],[[\'I-PER\',\'I-PER\']])[\'f1\']==0\n    for length in [1,2,3]:\n        for tags in itertools.product([\'O\',\'B-PER\',\'I-PER\',\'B-LOC\',\'I-LOC\'],repeat=length):\n            metrics([list(tags)],[list(tags)])\n    evidence[\'strict_metrics\']=\'known exact/partial/type/missing/repeated/illegal examples; decoder cross-checked with strict seqeval\'\n    crf=LinearCRF(3);e=torch.randn(2,3,3,requires_grad=True);tags=torch.tensor([[0,1,2],[2,1,0]]);mask=torch.tensor([[1,1,1],[1,1,0]],dtype=torch.bool)\n    nll=crf.nll(e,tags,mask);paths=crf.decode(e,mask);manual=[]\n    for b,n in enumerate([3,2]):\n        scores=[];ps=list(itertools.product(range(3),repeat=n))\n        for path in ps:\n            s=crf.start[path[0]]+crf.end[path[-1]]+sum(e[b,i,t] for i,t in enumerate(path))\n            s=s+sum(crf.transitions[path[i-1],path[i]] for i in range(1,n));scores.append(s)\n        stacked=torch.stack(scores);truth=tuple(tags[b,:n].tolist());manual.append(torch.logsumexp(stacked,0)-stacked[ps.index(truth)])\n        assert paths[b]==list(ps[int(stacked.argmax())])\n    assert torch.allclose(nll,torch.stack(manual).mean(),atol=1e-5)\n    nll.backward();assert e.grad.abs().sum()>0 and crf.transitions.grad.abs().sum()>0\n    padded=e.detach().clone();padded[1,2]=1e4\n    assert torch.allclose(crf.nll(padded,tags,mask),nll.detach()) and crf.decode(padded,mask)==paths\n    try:crf.nll(e.detach(),tags.masked_fill(~mask,-100),mask);raise AssertionError(\'-100 allowed\')\n    except AssertionError as exc:assert \'-100\' in str(exc)\n    assert crf.decode(e[:,:1].detach(),mask[:,:1]) and torch.isfinite(crf.nll(e[:,:1],tags[:,:1],mask[:,:1]))\n    evidence[\'crf\']=\'brute-force partition, exact likelihood/Viterbi, start/end transitions, length1, padding invariance, -100 rejection, finite real gradients\'\n    rows=read_json(root/\'prepared/train.json\')[:3]\n    for name in [\'lstm\',\'bilstm\',\'bilstm_crf\']:\n        cfg={\'model\':name,\'n_words\':len(vocab[\'words\']),\'n_chars\':len(vocab[\'chars\']),\'n_labels\':len(labels)}\n        model=RecurrentNER(cfg,np.load(root/\'prepared/embeddings.npy\')).eval()\n        a=collate_recurrent([rows[0]],vocab);b=collate_recurrent(rows,vocab)\n        with torch.no_grad():\n            ea=model(a[\'words\'],a[\'chars\'],a[\'mask\']);eb=model(b[\'words\'],b[\'chars\'],b[\'mask\'])\n        assert torch.allclose(ea[0],eb[0,:ea.shape[1]],atol=2e-6),name\n        model.train();ee=model(b[\'words\'],b[\'chars\'],b[\'mask\']);loss=model.loss(ee,b[\'tags\'],b[\'mask\']);loss.backward()\n        assert model.char.weight.grad.abs().sum()>0 and model.encoder.weight_ih_l0.grad.abs().sum()>0\n        assert model.word.weight.grad[0].abs().sum()==0\n        unknown=collate_recurrent([{\'tokens\':[\'ZzyzxNovelName\']}],vocab)\n        assert unknown[\'words\'][0,0]==1 and (unknown[\'chars\']!=0).any()\n    evidence[\'recurrent\']=\'all 3: padding-invariant eval emissions, masked loss, character/LSTM gradients, zero pad gradient, unknown-word characters\'\n    tok=AutoTokenizer.from_pretrained(root/\'assets/distilbert\',local_files_only=True,use_fast=True)\n    sample=[{\'tokens\':[\'Alice\',\',\',\'Supercalifragilisticexpialidocious\',\'New\',\'York\',\'.\'],\'tags\':[\'B-PER\',\'O\',\'O\',\'B-LOC\',\'I-LOC\',\'O\']},\n            {\'tokens\':[\'London\'],\'tags\':[\'B-LOC\']}]\n    feats,mapping=aligned_features(sample,tok,labels,256)\n    assert len(tok(sample[0][\'tokens\'][2],add_special_tokens=False)[\'input_ids\'])>1\n    coll=DataCollatorForTokenClassification(tok)(feats)\n    assert (coll[\'labels\'][coll[\'attention_mask\']==0]==-100).all()\n    ids=[]\n    for f in feats:\n        assert f[\'labels\'][0]==-100 and f[\'labels\'][-1]==-100\n        ids.append([t if t!=-100 else 0 for t in f[\'labels\']])\n    assert reconstruct(ids,mapping,sample,labels)==[r[\'tags\'] for r in sample]\n    assert sum(t!=-100 for f in feats for t in f[\'labels\'])==7\n    long=[{\'tokens\':[\'Alice\']*700,\'tags\':[\'B-PER\']+[\'I-PER\']*699}]\n    ff,mm=aligned_features(long,tok,labels,256);assert len(ff)>1\n    pp=reconstruct([[t if t!=-100 else 0 for t in f[\'labels\']] for f in ff],mm,long,labels)\n    assert pp==[long[0][\'tags\']] and len(strict_spans(pp[0]))==1\n    evidence[\'alignment\']=\'first subword only, special/padded -100, split word, punctuation, one output per word, 700-word entity reconstructed across chunks\'\n    text=\'Alice,  New York!\\nAlice & <script>\'\n    sentences=text_sentences(text);words=[w for s in sentences for w in s]\n    assert all(text[a:b]==w for w,a,b in words)\n    tags=[\'B-PER\' if w==\'Alice\' else \'B-LOC\' if w==\'New\' else \'I-LOC\' if w==\'York\' else \'O\' for w,a,b in words]\n    spans=highlight_spans(text,words,tags)\n    assert [s[\'text\'] for s in spans]==[\'Alice\',\'New York\',\'Alice\']\n    from app import colored_html\n    assert \'&lt;script&gt;\' in colored_html(text,spans)\n    evidence[\'offsets\']=\'original punctuation, repeated entities, multiword whitespace, linebreak offsets and HTML escaping\'\n    # Ensure the BiLSTM and CRF encoder structures match exactly.\n    a=RecurrentNER({\'model\':\'bilstm\',\'n_words\':len(vocab[\'words\']),\'n_chars\':len(vocab[\'chars\']),\'n_labels\':len(labels)})\n    b=RecurrentNER({**a.cfg,\'model\':\'bilstm_crf\'})\n    assert {k:tuple(v.shape) for k,v in a.state_dict().items()}=={k:tuple(v.shape) for k,v in b.state_dict().items() if not k.startswith(\'crf.\')}\n    evidence[\'matched_encoders\']=True\n    write_json(root/\'reports/unit_verification.json\',evidence);print(json.dumps(evidence,indent=2),flush=True)\n    return evidence\n\ndef scheduler_tests(root):\n    from scheduler import run_queue,worker_env,completed\n    root=Path(root)/\'scheduler_checks\'\n    if root.exists():__import__(\'shutil\').rmtree(root)\n    root.mkdir(parents=True,exist_ok=True)\n    script=root/\'fake.py\'\n    script.write_text("import os,sys,time\\nfrom pathlib import Path\\nPath(sys.argv[1]).write_text(str(os.getpid()))\\nassert os.environ[\'CUDA_VISIBLE_DEVICES\']==sys.argv[2]\\ntime.sleep(float(sys.argv[3]))\\nsys.exit(int(sys.argv[4]))\\n")\n    def cmd(job,dev):return [sys.executable,str(script.resolve()),str((root/(job+\'.pid\')).resolve()),dev,\'0.3\' if job==\'fail\' else \'1.0\',\'7\' if job==\'fail\' else \'0\']\n    two=run_queue([\'a\',\'b\',\'c\'],[\'0\',\'1\'],cmd,root/\'two\',poll=.05)\n    starts={e[\'model\']:e[\'time\'] for e in two if e[\'event\']==\'start\'};ends={e[\'model\']:e[\'time\'] for e in two if e[\'event\']==\'exit\'}\n    assert starts[\'a\']<ends[\'b\'] and starts[\'b\']<ends[\'a\']\n    single=run_queue([\'a\',\'b\'],[\'0\'],cmd,root/\'single\',poll=.05)\n    assert [e[\'event\'] for e in single]==[\'start\',\'exit\',\'start\',\'exit\']\n    try:run_queue([\'fail\',\'peer\',\'never\'],[\'0\',\'1\'],cmd,root/\'failure\',poll=.05);raise AssertionError(\'Expected failure\')\n    except RuntimeError:pass\n    failed=read_json(root/\'failure/schedule_events.json\');assert any(e[\'event\']==\'cancel\' and e[\'model\']==\'peer\' for e in failed)\n    assert not any(e[\'event\']==\'start\' and e[\'model\']==\'never\' for e in failed)\n    for p in [\'fail\',\'peer\']:\n        pid=int((root/(p+\'.pid\')).read_text())\n        try:os.kill(pid,0);raise AssertionError(\'Leaked process\')\n        except ProcessLookupError:pass\n    # Explicit SIGINT cancellation exercises the same parent cleanup used by notebook interruption.\n    wrapper=root/\'interrupt_wrapper.py\'\n    wrapper.write_text(\'import sys\\nfrom pathlib import Path\\nsys.path.insert(0,\'+repr(str(Path(__file__).parent.resolve()))+\')\\nfrom scheduler import run_queue\\nr=Path(\'+repr(str(root.resolve()))+\')\\nscript=r/"fake.py"\\nrun_queue(["interrupt_a","interrupt_b"],["0","1"],lambda job,dev:[sys.executable,str(script),str(r/(job+".pid")),dev,"20","0"],r/"interrupt",poll=.05)\\n\')\n    with open(root/\'interrupt_parent.log\',\'w\') as log:\n        parent=subprocess.Popen([sys.executable,str(wrapper)],stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n        deadline=time.time()+8\n        while not all((root/(n+\'.pid\')).exists() for n in [\'interrupt_a\',\'interrupt_b\']):\n            if time.time()>deadline:parent.kill();raise RuntimeError(\'Interrupt simulation failed to start\')\n            time.sleep(.05)\n        import signal\n        parent.send_signal(signal.SIGINT);parent.wait(timeout=8)\n    interrupt=read_json(root/\'interrupt/schedule_events.json\')\n    assert sum(e[\'event\']==\'cancel\' for e in interrupt)==2\n    for name in [\'interrupt_a\',\'interrupt_b\']:\n        pid=int((root/(name+\'.pid\')).read_text())\n        try:os.kill(pid,0);raise AssertionError(\'Interrupted child leaked\')\n        except ProcessLookupError:pass\n    env=worker_env(\'1\');assert env[\'CUDA_VISIBLE_DEVICES\']==\'1\' and \'RANK\' not in env\n    cache=root/\'cache\';folder=cache/\'models\'/\'fixture\';folder.mkdir(parents=True)\n    item=folder/\'weights.bin\';item.write_bytes(b\'known fixture\')\n    write_json(folder/\'done.json\',{\'experiment_id\':\'known\',\'files\':{\'weights.bin\':file_hash(item)}})\n    assert completed(cache,\'fixture\',\'known\')\n    try:completed(cache,\'fixture\',\'changed\');raise AssertionError(\'Identity mismatch accepted\')\n    except RuntimeError:pass\n    item.write_bytes(b\'corrupt\')\n    try:completed(cache,\'fixture\',\'known\');raise AssertionError(\'Corruption accepted\')\n    except RuntimeError:pass\n    out={\'type\':\'real CPU subprocess scheduler with simulated GPU identifiers; not real CUDA\',\n         \'overlap\':True,\'single_device_serial\':True,\'failure_cancels_peer\':True,\'queue_stops_on_failure\':True,\'no_leaked_processes\':True,\'environment_isolation\':True,\'explicit_sigint_cancellation\':True,\'cache_identity_and_corruption_rejection\':True}\n    write_json(root.parent/\'reports/scheduler_verification.json\',out);return out\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);a=p.parse_args();run_tests(a.root);scheduler_tests(a.root)\n', 'worker.py': '"""Fresh isolated process only: scheduler sets CUDA_VISIBLE_DEVICES before imports."""\nimport os,sys,json,time,platform,random,shutil,subprocess\nfrom pathlib import Path\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\',\'false\');os.environ.setdefault(\'WANDB_DISABLED\',\'true\')\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'2\');os.environ.setdefault(\'MKL_NUM_THREADS\',\'2\')\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\nfrom common import *\nfrom models import *\n\ndef seed_all(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\ndef device_setup():\n    torch.set_num_threads(2);torch.set_num_interop_threads(1)\n    n=torch.cuda.device_count(); visible=os.environ.get(\'CUDA_VISIBLE_DEVICES\',\'unset\')\n    assert n<=1, f\'Worker isolation failed: sees {n} GPUs; CVD={visible}\'\n    device=torch.device(\'cuda:0\' if n else \'cpu\')\n    print(json.dumps({\'pid\':os.getpid(),\'CUDA_VISIBLE_DEVICES\':visible,\'visible_cuda_devices\':n,\'device\':str(device),\n                      \'torch\':torch.__version__,\'cuda_build\':torch.version.cuda,\'gpu_name\':torch.cuda.get_device_name(0) if n else None}),flush=True)\n    return device\n\ndef move(batch,device):return {k:v.to(device) for k,v in batch.items()}\n\ndef recurrent_predictions(model,rows,vocab,device,batch_size=32):\n    loader=DataLoader(rows,batch_size=batch_size,shuffle=False,collate_fn=lambda r:collate_recurrent(r,vocab),num_workers=0)\n    out=[];losses=[];model.eval();start=time.perf_counter()\n    if device.type==\'cuda\':torch.cuda.synchronize()\n    with torch.no_grad():\n        for b in loader:\n            b=move(b,device);e=model(b[\'words\'],b[\'chars\'],b[\'mask\']);losses.append(float(model.loss(e,b[\'tags\'],b[\'mask\'])))\n            out.extend([[vocab[\'labels\'][t] for t in row] for row in model.decode(e,b[\'mask\'])])\n    if device.type==\'cuda\':torch.cuda.synchronize()\n    elapsed=time.perf_counter()-start\n    assert len(out)==len(rows) and all(len(a)==len(r[\'tokens\']) for a,r in zip(out,rows))\n    return out,elapsed,float(np.mean(losses))\n\ndef base_rows(root,mode):\n    train=read_json(root/\'prepared/train.json\');val=read_json(root/\'prepared/validation.json\')\n    if mode==\'smoke\':train=train[:32];val=val[:16]\n    return train,val\n\ndef recurrent(root,model_name,mode,out,device):\n    vocab=read_json(root/\'prepared/vocab.json\');train,val=base_rows(root,mode)\n    cfg={\'model\':model_name,\'n_words\':len(vocab[\'words\']),\'n_chars\':len(vocab[\'chars\']),\'n_labels\':len(vocab[\'labels\'])}\n    model=RecurrentNER(cfg,np.load(root/\'prepared/embeddings.npy\')).to(device)\n    params=sum(p.numel() for p in model.parameters() if p.requires_grad)\n    optim=torch.optim.Adam(model.parameters(),lr=.001)\n    use_amp=device.type==\'cuda\';scaler=torch.amp.GradScaler(\'cuda\',enabled=use_amp)\n    loader=DataLoader(train,batch_size=32,shuffle=True,num_workers=0,collate_fn=lambda r:collate_recurrent(r,vocab),generator=torch.Generator().manual_seed(42))\n    best=-1.;bad=0;history=[];gradient_evidence={};start=time.perf_counter();best_epoch=None\n    for epoch in range(1,(1 if mode==\'smoke\' else 15)+1):\n        model.train();loss_sum=0.;count=0\n        for step,b in enumerate(loader):\n            b=move(b,device);optim.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):\n                e=model(b[\'words\'],b[\'chars\'],b[\'mask\']);loss=model.loss(e,b[\'tags\'],b[\'mask\'])\n            assert torch.isfinite(loss);scaler.scale(loss).backward();scaler.unscale_(optim)\n            if not gradient_evidence:\n                gradient_evidence={name:float(p.grad.detach().float().norm()) for name,p in model.named_parameters() if p.grad is not None}\n                assert gradient_evidence[\'encoder.weight_ih_l0\']>0 and gradient_evidence[\'char.weight\']>0\n                if model.crf:assert gradient_evidence[\'crf.transitions\']>0\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.);scaler.step(optim);scaler.update()\n            with torch.no_grad():model.word.weight[0].zero_();model.char.weight[0].zero_()\n            loss_sum+=float(loss.detach())*len(b[\'words\']);count+=len(b[\'words\'])\n            if step%100==0:print(f\'{model_name} epoch {epoch} step {step}/{len(loader)} loss={float(loss.detach()):.4f}\',flush=True)\n        pred,latency,val_loss=recurrent_predictions(model,val,vocab,device)\n        score=metrics([r[\'tags\'] for r in val],pred)\n        history.append({\'epoch\':epoch,\'train_loss\':loss_sum/count,\'val_loss\':val_loss,\'val_f1\':score[\'f1\'],\'validation_seconds\':latency})\n        print(model_name,json.dumps(history[-1]),flush=True)\n        if score[\'f1\']>best:\n            best=score[\'f1\'];bad=0;best_epoch=epoch\n            torch.save({\'state\':{k:v.cpu() for k,v in model.state_dict().items()},\'config\':cfg},out/\'best.pt\')\n        else:bad+=1\n        if bad>=3:break\n    train_seconds=time.perf_counter()-start\n    model=load_recurrent(out/\'best.pt\',device)\n    pred,latency,val_loss=recurrent_predictions(model,val,vocab,device)\n    reloaded=load_recurrent(out/\'best.pt\',device)\n    pred2,_,_=recurrent_predictions(reloaded,val[:2],vocab,device);assert pred2==pred[:2]\n    write_json(out/\'validation_predictions.json\',pred)\n    return {\'parameters\':params,\'trainable_parameters\':params,\'history\':history,\'best_epoch\':best_epoch,\n            \'validation\':metrics([r[\'tags\'] for r in val],pred),\'validation_seconds\':latency,\n            \'train_seconds\':train_seconds,\'gradient_evidence\':gradient_evidence,\'reload_exact\':True,\n            \'mixed_precision\':\'fp16 autocast + scaler; CRF sequence computations float32\' if use_amp else \'float32 CPU\'}\n\nclass FeatureDataset(torch.utils.data.Dataset):\n    def __init__(self,features):self.features=features\n    def __len__(self):return len(self.features)\n    def __getitem__(self,i):return self.features[i]\n\ndef transformer_predict(model,tok,records,labels,length,device,batch_size=16):\n    from transformers import DataCollatorForTokenClassification\n    feats,mapping=aligned_features(records,tok,labels,length)\n    loader=DataLoader(FeatureDataset(feats),batch_size=batch_size,collate_fn=DataCollatorForTokenClassification(tok),shuffle=False,num_workers=0)\n    chunks=[];model.eval();start=time.perf_counter()\n    if device.type==\'cuda\':torch.cuda.synchronize()\n    with torch.no_grad():\n        for b in loader:\n            b={k:v.to(device) for k,v in b.items() if k!=\'labels\'}\n            logits=model(**b).logits;chunks.extend(logits.argmax(-1).cpu().tolist())\n    if device.type==\'cuda\':torch.cuda.synchronize()\n    elapsed=time.perf_counter()-start\n    return reconstruct(chunks,mapping,records,labels),elapsed\n\ndef transformer(root,mode,out,device):\n    from transformers import (AutoTokenizer,AutoModelForTokenClassification,Trainer,TrainingArguments,\n        DataCollatorForTokenClassification,EarlyStoppingCallback,TrainerCallback)\n    from prepare import fetch\n    lock=read_json(Path(__file__).with_name(\'source_lock.json\'))\n    info=lock[\'files\'][\'distilbert/model.safetensors\']\n    fetch(info[\'url\'],root/\'assets/distilbert/model.safetensors\',info[\'sha256\'])\n    tok=AutoTokenizer.from_pretrained(root/\'assets/distilbert\',use_fast=True,local_files_only=True)\n    vocab=read_json(root/\'prepared/vocab.json\');labels=vocab[\'labels\'];length=read_json(root/\'prepared/length_policy.json\')[\'max_subwords\']\n    train,val=base_rows(root,mode)\n    ft,mt=aligned_features(train,tok,labels,length);fv,mv=aligned_features(val,tok,labels,length)\n    model=AutoModelForTokenClassification.from_pretrained(root/\'assets/distilbert\',local_files_only=True,\n        num_labels=len(labels),id2label=dict(enumerate(labels)),label2id={t:i for i,t in enumerate(labels)},attn_implementation=\'eager\')\n    params=sum(p.numel() for p in model.parameters());assert all(p.requires_grad for p in model.distilbert.parameters())\n    # Observe a real encoder gradient on Trainer\'s first backward pass; no additional training path.\n    gradient={}\n    handle=model.distilbert.transformer.layer[0].attention.q_lin.weight.register_hook(lambda g:gradient.update({\'encoder_q_gradient_norm\':float(g.float().norm())}))\n    def compute(p):\n        logits=p.predictions[0] if isinstance(p.predictions,tuple) else p.predictions\n        pred=reconstruct(np.argmax(logits,axis=-1),mv,val,labels)\n        return {\'f1\':metrics([r[\'tags\'] for r in val],pred)[\'f1\']}\n    class FiniteCallback(TrainerCallback):\n        def on_log(self,args,state,control,logs=None,**kwargs):\n            if logs:print(\'distilbert\',json.dumps(logs),flush=True)\n    args=TrainingArguments(output_dir=str(out/\'trainer\'),learning_rate=2e-5,per_device_train_batch_size=16,\n        per_device_eval_batch_size=16,gradient_accumulation_steps=1,num_train_epochs=1 if mode==\'smoke\' else 5,\n        weight_decay=.01,warmup_ratio=.1,eval_strategy=\'epoch\',save_strategy=\'epoch\',logging_strategy=\'steps\',logging_steps=50,\n        load_best_model_at_end=True,metric_for_best_model=\'f1\',greater_is_better=True,save_total_limit=1,\n        save_only_model=True,fp16=device.type==\'cuda\',bf16=False,max_grad_norm=1.,report_to=[],seed=42,data_seed=42,\n        dataloader_num_workers=0,dataloader_pin_memory=device.type==\'cuda\',use_cpu=device.type==\'cpu\',disable_tqdm=False)\n    assert args.n_gpu<=1,\'Trainer must see exactly its assigned one GPU\'\n    trainer=Trainer(model=model,args=args,train_dataset=FeatureDataset(ft),eval_dataset=FeatureDataset(fv),\n        processing_class=tok,data_collator=DataCollatorForTokenClassification(tok),compute_metrics=compute,\n        callbacks=[EarlyStoppingCallback(early_stopping_patience=2),FiniteCallback()])\n    start=time.perf_counter();train_output=trainer.train();elapsed=time.perf_counter()-start;handle.remove()\n    assert gradient.get(\'encoder_q_gradient_norm\',0)>0,\'No actual transformer encoder gradient\'\n    assert all(np.isfinite(x) for x in gradient.values())\n    bestdir=out/\'best\';trainer.save_model(bestdir);tok.save_pretrained(bestdir)\n    pred,latency=transformer_predict(trainer.model,tok,val,labels,length,device)\n    reload_model=AutoModelForTokenClassification.from_pretrained(bestdir,local_files_only=True).to(device).eval()\n    pred2,_=transformer_predict(reload_model,tok,val[:2],labels,length,device);assert pred2==pred[:2]\n    write_json(out/\'validation_predictions.json\',pred)\n    history=[]\n    for row in trainer.state.log_history:\n        if \'eval_f1\' in row:history.append({\'epoch\':row.get(\'epoch\'),\'val_f1\':row[\'eval_f1\'],\'val_loss\':row.get(\'eval_loss\')})\n        elif \'loss\' in row:history.append({\'epoch\':row.get(\'epoch\'),\'train_loss\':row[\'loss\']})\n        elif \'train_loss\' in row:history.append({\'epoch\':row.get(\'epoch\'),\'train_loss\':row[\'train_loss\']})\n    write_json(out/\'trainer_state_evidence.json\',__import__(\'dataclasses\').asdict(trainer.state))\n    best_checkpoint=trainer.state.best_model_checkpoint\n    shutil.rmtree(out/\'trainer\',ignore_errors=True) # completed checkpoints only; no optimizer resume claim\n    return {\'parameters\':params,\'trainable_parameters\':params,\'history\':history,\'best_checkpoint\':best_checkpoint,\n        \'validation\':metrics([r[\'tags\'] for r in val],pred),\'validation_seconds\':latency,\'train_seconds\':elapsed,\n        \'gradient_evidence\':gradient,\'reload_exact\':True,\'mixed_precision\':\'fp16 Trainer\' if device.type==\'cuda\' else \'float32 CPU\'}\n\ndef run(root,model,mode,identity):\n    root=Path(root);out=root/\'models\'/model;out.mkdir(parents=True,exist_ok=True)\n    seed_all(42);device=device_setup()\n    if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n    result=transformer(root,mode,out,device) if model==\'distilbert\' else recurrent(root,model,mode,out,device)\n    result.update({\'model\':model,\'mode\':mode,\'experiment_id\':identity,\'physical_gpu\':os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n        \'visible_gpus\':torch.cuda.device_count(),\'device\':str(device),\'cuda_max_allocated_bytes\':torch.cuda.max_memory_allocated() if device.type==\'cuda\' else None,\n        \'cuda_max_reserved_bytes\':torch.cuda.max_memory_reserved() if device.type==\'cuda\' else None,\n        \'checkpoint_bytes\':sum(p.stat().st_size for p in out.rglob(\'*\') if p.is_file() and (p.suffix==\'.pt\' or p.suffix==\'.safetensors\')),\n        \'cpu_peak_rss_bytes\':__import__(\'resource\').getrusage(__import__(\'resource\').RUSAGE_SELF).ru_maxrss*1024,\n        \'timing_note\':\'Concurrent training; validation timing is synchronized, one pass, batch32 RNN/batch16 transformer; not isolated benchmarks\',\n        \'optimizer_resume\':False})\n    write_json(out/\'result.json\',result)\n    files={str(p.relative_to(out)):file_hash(p) for p in sorted(out.rglob(\'*\')) if p.is_file() and p.name!=\'done.json\'}\n    write_json(out/\'done.json\',{\'experiment_id\':identity,\'files\':files})\n    print(\'COMPLETED\',model,flush=True)\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--model\',required=True);p.add_argument(\'--mode\',default=\'full\');p.add_argument(\'--identity\',required=True)\n    a=p.parse_args();run(a.root,a.model,a.mode,a.identity)\n'}
for name, content in EMBEDDED_FILES.items():
    target = SOURCE / name
    target.write_text(content, encoding="utf-8")
print("Embedded files:", ", ".join(EMBEDDED_FILES))


## Install once, validate in a fresh process

An isolated virtual environment exposes only the host’s PyTorch/CUDA components, keeping unrelated preloaded audio/vision packages out of the text runtime. It installs the tested pinned text package set and verifies fresh imports and `pip check`. Same-session reruns reuse this environment. A PyTorch or dependency error stops here with its diagnostic, before training.

Do not use a Kaggle factory reset to apply installations. This path needs no restart. Host PyTorch below 2.4 is rejected with a request to use a current Kaggle image; it is never silently replaced.


In [ ]:
def run_process(command, *, env=None, log_name="stage.log"):
    env=dict(os.environ if env is None else env);env['MPLBACKEND']='Agg'
    ROOT.mkdir(parents=True, exist_ok=True)
    log_path = ROOT / log_name
    print("Log:", log_path, flush=True)
    with log_path.open("w") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=env, start_new_session=True)
        try:
            for line in process.stdout:
                log.write(line); log.flush()
                print(line, end="", flush=True)
            code = process.wait()
            if code:
                raise RuntimeError(f"Stage exited {code}; inspect {log_path}. Completed compatible models remain available.")
        except BaseException:
            if process.poll() is None:
                # Signal only this spawned project process group. Scheduler cancels its own workers.
                os.killpg(process.pid, signal.SIGINT)
                try:
                    process.wait(timeout=12)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGTERM)
                    process.wait(timeout=5)
            raise

run_process([sys.executable, str(SOURCE / "bootstrap.py"), str(RUNTIME), str(SOURCE)],
            log_name="runtime_install.log")
PYTHON = RUNTIME / "bin" / "python"
assert PYTHON.exists()


## Observed cloud audit and fixed experiment policy

The reference below was generated from downloaded data, not guessed. Kaggle repeats the complete preparation and compares counts, tags, entity counts, tokenizer lengths, canonical sentence fingerprints, training vocabulary fingerprint, and the task embedding matrix checksum against it. Every external asset has a recorded SHA-256. GloVe additionally matches its published MD5.

| Split | Sentences | Original words | Longest cased-tokenizer sequence, including special tokens |
|---|---:|---:|---:|
| Train | 14,041 | 203,621 | 173 |
| Validation | 3,250 | 51,362 | 151 |
| Test | 3,453 | 46,435 | 148 |

The mirror’s tokens and raw NER tags match the upstream corpus ZIP in every official split. Actual Parquet metadata supplies numeric tag IDs. Observed labels are already IOB2, so **zero conversions** were required. A deterministic IOB1→IOB2 converter and conversion ledger remain in the implementation. Only gold normalization is allowed; predictions receive no BIO repair.

There are 129 distinct repeated sentence keys between train/validation, 78 between train/test, and 25 between validation/test. Within-split duplicates and conflicting annotations are also recorded. **No rows are deduplicated.** These overlap findings are a benchmark limitation.

The 23,623 training word types create a 23,625-row matrix including PAD/UNK. GloVe-50 supplies 21,009 word rows (88.93% of training types; 97.73% of training tokens). Word/character vocabularies are training-only. Original casing is retained in character features; lowercase is used only for GloVe lookup. The train maximum character length is 61. Character clipping on held-out/app inputs is explicit and audited.

A **256-subword** cap covers all observed benchmark sentences. A tested, label-independent whole-word chunk/reconstruction policy covers longer synthetic/app inputs. Special, padded, and non-first subword targets are -100. Recurrent sequences use packed dynamic padding with no word truncation. POS/chunk fields are audited but never supplied as predictive features.

Training uses seed 42; one recurrent layer, hidden size 128 per direction, character embeddings 30 and CNN output 50, dropout .3, Adam .001, batch32, maximum15 epochs/patience3. DistilBERT uses AdamW 2e-5, batch16, accumulation1, maximum5 epochs/patience2, warmup .1 and weight decay .01. Gradient clipping is 1.0. CUDA uses supported fp16; CRF likelihood and Viterbi scores are float32. CPU checks use float32. No batch size or learning rate scales with GPU count. No class weighting, resampling, or search is applied.

The best checkpoint uses strict IOB2 entity micro F1. Architectures within **0.001 absolute F1** of the best prefer fewer trainable parameters, then measured validation inference seconds, then stable model name. The rule and winner are persisted **before** final test inference. CRF is unconstrained and may improve or worsen boundaries; both outcomes are reported.


## Prepare → bounded verification → train → frozen final evaluation → export

Preparation verifies provenance, file checksums, schema, source split identity, IOB labels, word/subword alignment, OOV/embedding coverage, lengths and overlap. Full mode audits test structure but **never uses test predictions for preparation, tuning, early stopping, or selection**.

Before expensive training, bounded tests exercise masked losses, finite CRF likelihood/gradients and exhaustive tiny-path Viterbi checks, exact-span scoring, offsets, long-input reconstruction, scheduler overlap, cancellation, cache rejection and isolation. A separate tiny real train/validation-only run trains/reloads all four architectures and verifies the Gradio export. It cannot produce final-quality claims.

On two GPUs, one DistilBERT worker starts alongside one recurrent worker; the next architecture takes the next free device. Every worker sees at most one GPU, including Trainer. One GPU serializes the queue. Failure cancels active workers and stops selection. Completed checkpoints are hash-verified and reusable; interrupted optimizers **restart**, with no optimizer-state resume claim.

A CPU-only machine can run `smoke`/`prep`; full mode stops after bounded checks and requests a Kaggle GPU. Logs show each assignment, epoch/step progress and failures. Original logs are retained even if notebook output is shortened.


In [ ]:
project_env = os.environ.copy()
project_env.update({"MPLBACKEND": "Agg", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2",
                    "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
                    "WANDB_DISABLED": "true", "HF_HUB_DISABLE_TELEMETRY": "1"})
run_process([str(PYTHON), str(SOURCE / "pipeline.py"), "--root", str(ROOT), "--mode", MODE],
            env=project_env, log_name="pipeline.log")


## Review the outputs

Full mode produces:

- **`ner_reports.zip`** — audit, strict per-type precision/recall/F1/support, errors, CRF corrections/regressions, figures, validation selection, environment, and verification evidence.
- **`ner_all_checkpoints.zip`** — all four best checkpoints, vocabularies, policy, experiment configuration, embedded source, reports and worker logs. No full corpus, GloVe resource, or optimizer-state resume files.
- **`ner_gradio_app.zip`** — selected trained model plus standalone Gradio application, exact inference preprocessing, tokenizer when applicable, labels, original-character span reconstruction, dependency list and instructions. It needs neither training data nor GloVe at startup. Hosting is a separate action.

Smoke archives use a `smoke_` prefix and are explicitly labeled as functional checks. `prep` emits audit reports without models. Archive integrity, required checkpoints and clean exported inference are checked before completion.

Extract the app ZIP, install its requirements in a fresh Python 3.11/3.12 environment and run `python app.py`. The app includes colored entities, a legend, examples, an entity/offset table, blank/no-entity handling, an explicit 20,000-character limit, and bounded inference queuing. The raw-text regex tokenizer is deterministic and preserves offsets, but can differ from CoNLL human tokenization, especially contractions and abbreviations. No real-time latency or model-quality claim is made before the trained run is measured.


In [ ]:
from IPython.display import display, Markdown, Image, FileLink
reports = ROOT / "reports"
audit = json.loads((reports / "data_audit.json").read_text())
rows = ["| Split | Sentences | Words | O prevalence | Word OOV |", "|---|---:|---:|---:|---:|"]
for name, info in audit["splits"].items():
    rows.append(f"| {name} | {info['sentences']:,} | {info['tokens']:,} | {info['o_prevalence']:.2%} | {info['word_oov_rate']:.2%} |")
display(Markdown("\n".join(rows)))
if (reports / "REPORT.md").exists():
    display(Markdown((reports / "REPORT.md").read_text()))
for name in ["eda.png", "learning_curves.png"]:
    if (reports / name).exists():
        display(Image(filename=str(reports / name)))
outputs = ROOT / "outputs"
if outputs.exists():
    for file in sorted(outputs.glob("*.zip")):
        print(file.name, f"{file.stat().st_size / 1024**2:.1f} MiB")
        display(FileLink(str(file)))
    display(FileLink(str(outputs / "archive_manifest.json")))
else:
    print("Preparation-only run. Reports:", reports)


## Verification and reruns

The historical executed notebook records successful two-T4 full training, isolated workers, fp16, final held-out evaluation and three output archives. The original cloud verification bundle predates that GPU run.

The raw-v2 update is verified separately by local CPU inference and regression checks. It changes raw-text processing, not trained weights or the benchmark token API. Read README.md for current results, limits and reproduction steps. Re-running this revised notebook creates a new experiment; it does not silently reuse old completed models. Keep generated outputs before ending a Kaggle session.

Interrupted optimization restarts; compatible completed checkpoints can be reused. No public hosting or automatic typing updates are implied.
